# RNN data preparation and experimental scenes

This notebook generates the complete **6,000-scene RNN training set** and the **106 original experimental scenes** used for RNN testing.

## Local folder layout

The notebook assumes this local structure:

```text
~/Downloads/Meta-control/
├── rnn_data_prep.ipynb
├── rnn_training.ipynb
└── physics_abstraction_master/
    ├── python/
    └── data/json/
        ├── experiment1/
        └── experiment2/
```

All generated outputs are written inside `~/Downloads/Meta-control/`.

## Shared naming contract

The training notebook is configured to read these exact outputs:

```text
~/Downloads/Meta-control/
├── rnn_training_data/
├── rnn_testing_data/
├── compressed_frame_cache_100x128/
└── ball_position_cache/
```

Cache IDs are built from scene paths relative to `~/Downloads/Meta-control`, so the data-prep and training notebooks use the same cache filenames.

## Training set

The notebook generates, in one pass:

- **2,900 five-container scenes**
- **2,500 four-container scenes**
- **600 one-line scenes**
- **6,000 scenes total**

For each randomized candidate, physics runs until the ball first touches either the **goal** or the **ground**. The candidate is rejected if neither event occurs within frames `0..599`. At the first terminal event, the ball is frozen at that position and **60 additional frozen frames** are appended.

Accepted scenes therefore have variable lengths. If the first event occurs at zero-based frame `e`, the cached sequence contains frames `0..e+60`, for a total of `e+61` frames.

For every accepted scene, the notebook preserves the original setting-specific randomization and validation rules, renders at **800 × 1024** in memory, downsamples with bilinear interpolation to **100 × 128**, writes a memory-mappable `uint8` frame cache with shape `(T, 3, 128, 100)`, and writes the matching ball-position cache. It does not save high-resolution frame PNGs or per-scene MP4s; only the first three accepted scenes of each training setting receive cache-based preview MP4s.

## Experimental test set

After training-scene generation, the notebook processes the original **48 Exp1 + 58 Exp2 = 106 experimental scenes** with the same event-stopping, 60-frame freeze, rendering, and cache pipeline. Their original geometry is preserved exactly.

`RESET_OUTPUT = True` clears the training metadata folder and shared cache folders before generating the 6,000 training scenes. `RESET_TEST_OUTPUT = True` later clears only the testing metadata and testing cache entries, leaving the training data intact.


In [ ]:
# ============================================================
# Configuration and imports
# ============================================================
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import copy
import json
import math
import os
import random
import shutil
import sys
import time
import traceback

# Headless/offscreen rendering defaults; safe for local execution.
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ.setdefault("PYGAME_HIDE_SUPPORT_PROMPT", "1")

import numpy as np
import pandas as pd

HOME = Path.home()
BASE_DIR = HOME / "Downloads" / "Meta-control"
PROJECT_ROOT = BASE_DIR / "physics_abstraction_master"

EXP1_JSON_DIR = PROJECT_ROOT / "data" / "json" / "experiment1"
EXP2_JSON_DIR = PROJECT_ROOT / "data" / "json" / "experiment2"

OUTROOT = BASE_DIR / "rnn_training_data"
FRAME_CACHE_DIR = BASE_DIR / "compressed_frame_cache_100x128"
BALL_CACHE_DIR = BASE_DIR / "ball_position_cache"

# WARNING: True removes all three output folders above before generation.
RESET_OUTPUT = True

# The RNN itself needs only FRAME_CACHE_DIR and BALL_CACHE_DIR.
SAVE_SIMULATION_CSV = True

FPS = 60

# Search for the first goal/ground event only within frames 0..599.
MAX_EVENT_SEARCH_FRAMES = 600

# After the first event frame, append exactly 60 additional frozen frames.
POST_EVENT_FREEZE_FRAMES = 60

RENDER_WIDTH = 800
RENDER_HEIGHT = 1024
IMAGE_W = 100
IMAGE_H = 128

EXP1_GOAL_Y_OFFSET = -40.0
EXP2_GOAL_Y_OFFSET = 0.0

TARGET_SCENE_COUNTS = {
    "five_containers": 2900,
    "four_containers": 2500,
    "one_line": 600,
}
TOTAL_REQUESTED_SCENES = sum(TARGET_SCENE_COUNTS.values())

ONE_LINE_RANDOM_SEED = 20260612
FOUR_CONTAINER_RANDOM_SEED = 20260518
FIVE_CONTAINER_RANDOM_SEED = 20260518

MAX_ATTEMPTS_PER_SETTING = 2_000_000

# Use up to four CPU cores for rendering/downsampling each cache.
CACHE_RENDER_WORKERS = min(4, os.cpu_count() or 1)

FULL_DISK_VERIFY_SAMPLE_SIZE = 20
PREVIEW_SCENES_PER_SETTING = 3
PREVIEW_VIDEO_WIDTH = 800
PREVIEW_VIDEO_HEIGHT = 1024

# One-line acceptance settings copied from the original generator.
MIN_LINE_CONTACT_CONSEC_FRAMES = 51
BALL_LINE_DISTANCE_PX = 45.0
MIN_SLIDE_TANGENTIAL_PX = 20.0
BALL_TOP_MARGIN = 40.0
BALL_SIDE_MARGIN = 30.0
BALL_X_FRACTION_MIN = 0.0
BALL_X_FRACTION_MAX = 1.0
BALL_Y_CLEARANCE_ABOVE_LINE_MIN = 30.0
BALL_Y_CLEARANCE_ABOVE_LINE_MAX = 70.0
MAX_BALL_POSITION_TRIES_PER_LINE = 400

PYTHON_DIR = PROJECT_ROOT / "python"
MOTION_DIR = PROJECT_ROOT / "motion_distributions"

for required in [BASE_DIR, PROJECT_ROOT, PYTHON_DIR, EXP1_JSON_DIR, EXP2_JSON_DIR]:
    if not required.exists():
        raise FileNotFoundError(
            f"Missing required path: {required}\n"
            "Expected physics_abstraction_master beside this notebook under "
            "~/Downloads/Meta-control, with python/ and both source JSON folders."
        )

for d in [PYTHON_DIR, PROJECT_ROOT, MOTION_DIR]:
    sys.path = [x for x in sys.path if x != str(d)]
for d in reversed([PYTHON_DIR, PROJECT_ROOT, MOTION_DIR]):
    if d.exists():
        sys.path.insert(0, str(d))

os.chdir(PROJECT_ROOT)
for module_name in ["objects", "scene"]:
    sys.modules.pop(module_name, None)

import objects
from scene import Scene

max_output_frames = MAX_EVENT_SEARCH_FRAMES + POST_EVENT_FREEZE_FRAMES
max_bytes_per_scene = max_output_frames * 3 * IMAGE_H * IMAGE_W
max_total_cache_gib = (
    max_bytes_per_scene * TOTAL_REQUESTED_SCENES / (1024 ** 3)
)

print("BASE_DIR:", BASE_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("objects:", objects.__file__)
print("EXP1 JSONs:", len(list(EXP1_JSON_DIR.glob("*.json"))))
print("EXP2 JSONs:", len(list(EXP2_JSON_DIR.glob("*.json"))))
print("OUTROOT:", OUTROOT)
print("FRAME_CACHE_DIR:", FRAME_CACHE_DIR)
print("BALL_CACHE_DIR:", BALL_CACHE_DIR)
print("CACHE_RENDER_WORKERS:", CACHE_RENDER_WORKERS)
print("Requested scenes:", TARGET_SCENE_COUNTS)
print(
    "Maximum possible raw .npy cache storage "
    f"(if every event occurs at frame 599): {max_total_cache_gib:.1f} GiB"
)


In [ ]:
# ============================================================
# Generic JSON and object-pool helpers
# ============================================================
class Config:
    """Adapter for project objects that expect config.model_dump()."""
    def __init__(self, **kwargs):
        self.kwargs = kwargs

    def model_dump(self):
        return dict(self.kwargs)


def as_xy(value, label="position"):
    original = value

    while isinstance(value, list) and len(value) == 1:
        value = value[0]

    if isinstance(value, dict):
        for key in ("position", "pos", "point", "location"):
            if key in value:
                value = value[key]
                break

    while isinstance(value, list) and len(value) == 1:
        value = value[0]

    if (
        isinstance(value, (list, tuple))
        and len(value) >= 2
        and isinstance(value[0], (int, float))
        and isinstance(value[1], (int, float))
    ):
        return float(value[0]), float(value[1])

    raise ValueError(f"Could not parse {label} as x,y. Got: {original!r}")


def parse_screen_size(data):
    width, height = 800, 1000
    value = data.get("screen_size")
    while isinstance(value, list) and len(value) == 1:
        value = value[0]
    if isinstance(value, list) and len(value) >= 2:
        width, height = int(value[0]), int(value[1])
    return width, height


def get_json_bottom_border(data):
    args = data.get("bottom_border_args")
    if isinstance(args, list) and len(args) >= 2:
        return (
            as_xy(args[0], "bottom_border_args[0]"),
            as_xy(args[1], "bottom_border_args[1]"),
            "json_bottom_border_args",
        )
    return (10.0, 990.0), (790.0, 990.0), "default_bottom_border"


def get_json_side_border_length(data, bottom_y):
    args = data.get("plinko_border_args", [])
    while isinstance(args, list) and len(args) == 1:
        args = args[0]
    if isinstance(args, (int, float)):
        return int(args), "json_plinko_border_args"
    if isinstance(args, list) and args and isinstance(args[0], (int, float)):
        return int(args[0]), "json_plinko_border_args"
    return int(bottom_y + 10), "fallback_bottom_y_plus_10"


LINE_KEYS = ["line_args", "lines_args", "line_arg", "lines", "Line_args", "Line"]


def get_line_args_from_json(data):
    for key in LINE_KEYS:
        if key in data:
            value = data[key]
            if value is None:
                return [], key
            if isinstance(value, list):
                return value, key
            return [value], key
    return [], None


def set_line_args_in_json(data, value, key="line_args", keep_empty_key=True):
    for old_key in LINE_KEYS:
        data.pop(old_key, None)
    if value or keep_empty_key:
        data[key or "line_args"] = copy.deepcopy(value)


def parse_line(line_arg, idx):
    original = line_arg
    while isinstance(line_arg, list) and len(line_arg) == 1:
        line_arg = line_arg[0]

    if isinstance(line_arg, dict):
        if "point_a" in line_arg and "point_b" in line_arg:
            point_a = as_xy(line_arg["point_a"], f"line_args[{idx}].point_a")
            point_b = as_xy(line_arg["point_b"], f"line_args[{idx}].point_b")
        elif "a" in line_arg and "b" in line_arg:
            point_a = as_xy(line_arg["a"], f"line_args[{idx}].a")
            point_b = as_xy(line_arg["b"], f"line_args[{idx}].b")
        elif "p1" in line_arg and "p2" in line_arg:
            point_a = as_xy(line_arg["p1"], f"line_args[{idx}].p1")
            point_b = as_xy(line_arg["p2"], f"line_args[{idx}].p2")
        else:
            raise ValueError(f"Line dict missing endpoints: {original!r}")
        angle = float(line_arg.get("angle", 0))
    elif isinstance(line_arg, list):
        if (
            len(line_arg) >= 2
            and isinstance(line_arg[0], (list, tuple, dict))
            and isinstance(line_arg[1], (list, tuple, dict))
        ):
            point_a = as_xy(line_arg[0], f"line_args[{idx}][0]")
            point_b = as_xy(line_arg[1], f"line_args[{idx}][1]")
            angle = (
                float(line_arg[2])
                if len(line_arg) > 2 and isinstance(line_arg[2], (int, float))
                else 0.0
            )
        elif len(line_arg) >= 4:
            point_a = float(line_arg[0]), float(line_arg[1])
            point_b = float(line_arg[2]), float(line_arg[3])
            angle = (
                float(line_arg[4])
                if len(line_arg) > 4 and isinstance(line_arg[4], (int, float))
                else 0.0
            )
        else:
            raise ValueError(f"Line list format not understood: {original!r}")
    else:
        raise ValueError(f"Unknown line format: {original!r}")

    return {
        "id": idx,
        "point_a": point_a,
        "point_b": point_b,
        "angle": angle,
        "raw": original,
    }


def parse_container(container_arg, idx):
    value = container_arg
    while isinstance(value, list) and len(value) == 1 and isinstance(value[0], dict):
        value = value[0]

    if isinstance(value, dict):
        pos = as_xy(value, f"container_args[{idx}].position")
        width = float(value.get("width", value.get("w", 80)))
        length = float(value.get("length", value.get("l", 0)))
        angle = float(value.get("angle", 0))
    elif isinstance(value, list):
        pos = as_xy(value[0], f"container_args[{idx}][0]")
        width = float(value[1]) if len(value) > 1 else 80.0
        length = float(value[2]) if len(value) > 2 else 0.0
        angle = float(value[3]) if len(value) > 3 else 0.0
    else:
        raise ValueError(f"Unknown container format: {container_arg!r}")

    return {
        "id": idx,
        "x": pos[0],
        "y": pos[1],
        "width": width,
        "length": length,
        "angle": angle,
        "raw": container_arg,
    }


def replace_first_xy_preserving_structure(value, new_xy):
    x, y = float(new_xy[0]), float(new_xy[1])

    if isinstance(value, dict):
        out = copy.deepcopy(value)
        for key in ("position", "pos", "point", "location"):
            if key in out:
                old = out[key]
                out[key] = [[x, y]] if isinstance(old, list) and len(old) == 1 else [x, y]
                return out
        raise ValueError(f"Could not find a position key in ball_args: {value!r}")

    if isinstance(value, tuple):
        value = list(value)

    if isinstance(value, list):
        if (
            len(value) >= 2
            and isinstance(value[0], (int, float))
            and isinstance(value[1], (int, float))
        ):
            out = copy.deepcopy(value)
            out[0], out[1] = x, y
            return out
        if len(value) == 1:
            return [replace_first_xy_preserving_structure(value[0], (x, y))]

    raise ValueError(f"Could not replace x,y in ball_args structure: {value!r}")


def border_signature(data):
    bottom_a, bottom_b, _ = get_json_bottom_border(data)
    side_l, _ = get_json_side_border_length(data, max(bottom_a[1], bottom_b[1]))
    return (
        tuple(round(v, 6) for v in bottom_a),
        tuple(round(v, 6) for v in bottom_b),
        int(side_l),
        json.dumps(data.get("bottom_border_args", []), sort_keys=True),
        json.dumps(data.get("plinko_border_args", []), sort_keys=True),
    )


def choose_common_border(items):
    signatures = [border_signature(item["data"]) for item in items]
    most_common, count = Counter(signatures).most_common(1)[0]
    source_item = next(item for item in items if border_signature(item["data"]) == most_common)
    source = source_item["data"]
    return {
        "bottom_border_args": copy.deepcopy(
            source.get("bottom_border_args", [[10, 990], [790, 990]])
        ),
        "plinko_border_args": copy.deepcopy(source.get("plinko_border_args", [])),
        "source": source_item["path"].name,
        "matched": count,
        "total": len(items),
    }


def load_json_items(json_dir):
    items = []
    for path in sorted(json_dir.glob("*.json")):
        with open(path, "r") as f:
            data = json.load(f)
        as_xy(data.get("ball_args"), f"{path.name}: ball_args")
        as_xy(data.get("goal_args"), f"{path.name}: goal_args")
        items.append({"path": path, "data": data})
    if not items:
        raise RuntimeError(f"No usable JSON files found in {json_dir}")
    return items


def rebuild_objects_list(base_objects, has_containers, has_lines):
    values = list(base_objects) if isinstance(base_objects, list) else []
    filtered = []
    seen = set()
    for value in values:
        low = str(value).lower()
        if low in {"ball", "goal", "container", "line"}:
            continue
        if low not in seen:
            filtered.append(value)
            seen.add(low)

    result = ["Ball", "Goal"] + filtered
    if has_containers:
        result.append("Container")
    if has_lines:
        result.append("Line")
    return result


def canonical_scene_signature(data):
    lines, line_key = get_line_args_from_json(data)
    return json.dumps(
        {
            "ball_args": data.get("ball_args"),
            "goal_args": data.get("goal_args"),
            "container_args": data.get("container_args", []) or [],
            "line_args": lines or [],
            "line_key": line_key or "line_args",
            "bottom_border_args": data.get("bottom_border_args", []),
            "plinko_border_args": data.get("plinko_border_args", []),
        },
        sort_keys=True,
        separators=(",", ":"),
    )

In [ ]:
# ============================================================
# Build the source pools and preserve each generator's
# configuration-randomization strategy
# ============================================================
EXP1_ITEMS = load_json_items(EXP1_JSON_DIR)
EXP2_ITEMS = load_json_items(EXP2_JSON_DIR)

EXP1_BORDER = choose_common_border(EXP1_ITEMS)
EXP2_BORDER = choose_common_border(EXP2_ITEMS)


def build_exp1_pools(items):
    balls, goals, one_lines, five_containers = [], [], [], []
    for item in items:
        source = item["path"].name
        data = item["data"]

        balls.append({"source": source, "value": copy.deepcopy(data["ball_args"])})
        goals.append({"source": source, "value": copy.deepcopy(data["goal_args"])})

        lines, line_key = get_line_args_from_json(data)
        if isinstance(lines, list) and len(lines) == 1:
            try:
                parse_line(lines[0], 0)
                one_lines.append(
                    {
                        "source": source,
                        "value": copy.deepcopy(lines),
                        "key": line_key or "line_args",
                    }
                )
            except Exception:
                pass

        containers = data.get("container_args", []) or []
        if isinstance(containers, list) and len(containers) == 5:
            try:
                for idx, value in enumerate(containers):
                    parse_container(value, idx)
                five_containers.append(
                    {"source": source, "value": copy.deepcopy(containers)}
                )
            except Exception:
                pass

    if not balls or not goals or not one_lines or not five_containers:
        raise RuntimeError(
            "Experiment 1 must provide ball bundles, goal bundles, "
            "one-line layouts, and five-container layouts."
        )
    return balls, goals, one_lines, five_containers


def build_exp2_pools(items):
    balls, goals, four_containers = [], [], []
    for item in items:
        source = item["path"].name
        data = item["data"]

        balls.append({"source": source, "value": copy.deepcopy(data["ball_args"])})
        goals.append({"source": source, "value": copy.deepcopy(data["goal_args"])})

        containers = data.get("container_args", []) or []
        if isinstance(containers, list) and len(containers) == 4:
            try:
                for idx, value in enumerate(containers):
                    parse_container(value, idx)
                four_containers.append(
                    {"source": source, "value": copy.deepcopy(containers)}
                )
            except Exception:
                pass

    if not balls or not goals or not four_containers:
        raise RuntimeError(
            "Experiment 2 must provide ball bundles, goal bundles, "
            "and four-container layouts."
        )
    return balls, goals, four_containers


EXP1_BALLS, EXP1_GOALS, EXP1_ONE_LINES, EXP1_FIVE_CONTAINERS = build_exp1_pools(EXP1_ITEMS)
EXP2_BALLS, EXP2_GOALS, EXP2_FOUR_CONTAINERS = build_exp2_pools(EXP2_ITEMS)

EXISTING_EXP1_SIGNATURES = {
    canonical_scene_signature(item["data"]) for item in EXP1_ITEMS
}

print(
    "Experiment 1 pools:",
    len(EXP1_BALLS), "balls,",
    len(EXP1_GOALS), "goals,",
    len(EXP1_ONE_LINES), "one-line layouts,",
    len(EXP1_FIVE_CONTAINERS), "five-container layouts",
)
print(
    "Experiment 2 pools:",
    len(EXP2_BALLS), "balls,",
    len(EXP2_GOALS), "goals,",
    len(EXP2_FOUR_CONTAINERS), "four-container layouts",
)
print("Exp1 common border:", EXP1_BORDER)
print("Exp2 common border:", EXP2_BORDER)


def interpolate_line_y_at_x(line_info, x):
    x1, y1 = line_info["point_a"]
    x2, y2 = line_info["point_b"]
    if abs(x2 - x1) < 1e-6:
        return None
    alpha = (float(x) - x1) / (x2 - x1)
    return y1 + alpha * (y2 - y1)


def sample_ball_position_above_line(line_arg, rng, screen_width=800, screen_height=1024):
    line_info = parse_line(line_arg, 0)
    x1, y1 = map(float, line_info["point_a"])
    x2, y2 = map(float, line_info["point_b"])

    usable_xmin = max(float(BALL_SIDE_MARGIN), min(x1, x2))
    usable_xmax = min(float(screen_width) - float(BALL_SIDE_MARGIN), max(x1, x2))
    if usable_xmax <= usable_xmin:
        raise RuntimeError("line_x_range_too_small_for_guided_ball_sampling")

    last_reason = ""
    for _ in range(MAX_BALL_POSITION_TRIES_PER_LINE):
        frac = rng.uniform(BALL_X_FRACTION_MIN, BALL_X_FRACTION_MAX)
        x = x1 + frac * (x2 - x1)
        if not usable_xmin <= x <= usable_xmax:
            last_reason = "sampled_x_outside_usable_range"
            continue

        line_y = interpolate_line_y_at_x(line_info, x)
        if line_y is None:
            last_reason = "vertical_line"
            continue

        clearance = rng.uniform(
            BALL_Y_CLEARANCE_ABOVE_LINE_MIN,
            BALL_Y_CLEARANCE_ABOVE_LINE_MAX,
        )
        y = float(line_y) - clearance
        if y < BALL_TOP_MARGIN or y > float(screen_height) - BALL_TOP_MARGIN:
            last_reason = "sampled_y_outside_screen_margins"
            continue

        return float(x), float(y), line_info, float(line_y)

    raise RuntimeError(
        f"could_not_sample_ball_above_line_after_{MAX_BALL_POSITION_TRIES_PER_LINE}_tries: "
        f"{last_reason}"
    )


def make_one_line_candidate(scene_name, rng):
    # Strategy from the uploaded notebook:
    # pick a line first, use a raw ball bundle as a template, and replace only
    # its initial x,y with a guided random position above that line.
    base_item = rng.choice(EXP1_ITEMS)
    ball_pick = rng.choice(EXP1_BALLS)
    goal_pick = rng.choice(EXP1_GOALS)
    line_pick = rng.choice(EXP1_ONE_LINES)

    data = copy.deepcopy(base_item["data"])
    data["name"] = scene_name

    line_value = copy.deepcopy(line_pick["value"])
    set_line_args_in_json(data, line_value, line_pick["key"], keep_empty_key=True)

    lines, _ = get_line_args_from_json(data)
    screen_w, screen_h = parse_screen_size(data)
    ball_x, ball_y, parsed_line, line_y = sample_ball_position_above_line(
        lines[0], rng, screen_width=screen_w, screen_height=screen_h
    )

    data["ball_args"] = replace_first_xy_preserving_structure(
        copy.deepcopy(ball_pick["value"]),
        (ball_x, ball_y),
    )
    data["goal_args"] = copy.deepcopy(goal_pick["value"])
    data["container_args"] = []
    data["bottom_border_args"] = copy.deepcopy(EXP1_BORDER["bottom_border_args"])
    data["plinko_border_args"] = copy.deepcopy(EXP1_BORDER["plinko_border_args"])
    data["objects"] = rebuild_objects_list(data.get("objects", []), False, True)

    meta = {
        "base_schema_source": base_item["path"].name,
        "ball_args_source": ball_pick["source"],
        "goal_args_source": goal_pick["source"],
        "line_args_layout_source": line_pick["source"],
        "container_args_layout_source": "",
        "shared_border_source": EXP1_BORDER["source"],
        "guided_ball_x": ball_x,
        "guided_ball_y": ball_y,
        "guided_line_y_at_ball_x": line_y,
        "line_point_a": parsed_line["point_a"],
        "line_point_b": parsed_line["point_b"],
    }
    return data, meta


def make_four_container_candidate(scene_name, rng):
    # Strategy from the four-container .py file:
    # independently choose whole raw Exp2 ball, goal, and 4-container bundles.
    base_item = rng.choice(EXP2_ITEMS)
    ball_pick = rng.choice(EXP2_BALLS)
    goal_pick = rng.choice(EXP2_GOALS)
    container_pick = rng.choice(EXP2_FOUR_CONTAINERS)

    data = copy.deepcopy(base_item["data"])
    data["name"] = scene_name
    data["ball_args"] = copy.deepcopy(ball_pick["value"])
    data["goal_args"] = copy.deepcopy(goal_pick["value"])
    data["container_args"] = copy.deepcopy(container_pick["value"])
    set_line_args_in_json(data, [], "line_args", keep_empty_key=True)
    data["bottom_border_args"] = copy.deepcopy(EXP2_BORDER["bottom_border_args"])
    data["plinko_border_args"] = copy.deepcopy(EXP2_BORDER["plinko_border_args"])
    data["objects"] = rebuild_objects_list(data.get("objects", []), True, False)

    meta = {
        "base_schema_source": base_item["path"].name,
        "ball_args_source": ball_pick["source"],
        "goal_args_source": goal_pick["source"],
        "container_args_layout_source": container_pick["source"],
        "line_args_layout_source": "",
        "shared_border_source": EXP2_BORDER["source"],
    }
    return data, meta


def make_five_container_candidate(scene_name, rng):
    # Strategy from the five-container case in the Exp1 .py file:
    # independently choose whole raw Exp1 ball, goal, and 5-container bundles.
    base_item = rng.choice(EXP1_ITEMS)
    ball_pick = rng.choice(EXP1_BALLS)
    goal_pick = rng.choice(EXP1_GOALS)
    container_pick = rng.choice(EXP1_FIVE_CONTAINERS)

    data = copy.deepcopy(base_item["data"])
    data["name"] = scene_name
    data["ball_args"] = copy.deepcopy(ball_pick["value"])
    data["goal_args"] = copy.deepcopy(goal_pick["value"])
    data["container_args"] = copy.deepcopy(container_pick["value"])
    set_line_args_in_json(data, [], "line_args", keep_empty_key=False)
    data["bottom_border_args"] = copy.deepcopy(EXP1_BORDER["bottom_border_args"])
    data["plinko_border_args"] = copy.deepcopy(EXP1_BORDER["plinko_border_args"])
    data["objects"] = rebuild_objects_list(data.get("objects", []), True, False)

    meta = {
        "base_schema_source": base_item["path"].name,
        "ball_args_source": ball_pick["source"],
        "goal_args_source": goal_pick["source"],
        "container_args_layout_source": container_pick["source"],
        "line_args_layout_source": "",
        "shared_border_source": EXP1_BORDER["source"],
    }
    return data, meta

In [ ]:
# ============================================================
# Scene construction, validation, and event-stopped variable-length simulation
# ============================================================
def find_ball(scene_obj):
    for obj in scene_obj.objects:
        if getattr(obj, "name", "") == "Ball":
            return obj
    raise RuntimeError("No Ball object found.")


def build_scene_from_data(data, goal_y_offset):
    json_width, json_height = parse_screen_size(data)
    ball_start = as_xy(data["ball_args"], "ball_args")
    original_goal = as_xy(data["goal_args"], "goal_args")
    goal_pos = (original_goal[0], original_goal[1] + goal_y_offset)

    bottom_a, bottom_b, bottom_source = get_json_bottom_border(data)
    side_l, side_source = get_json_side_border_length(
        data, max(bottom_a[1], bottom_b[1])
    )

    scene_objects = [
        objects.Ball(Config(position=ball_start)),
        objects.Goal(Config(position=goal_pos)),
    ]

    if hasattr(objects, "LeftBorder"):
        try:
            scene_objects.append(objects.LeftBorder(l=side_l))
        except TypeError:
            scene_objects.append(objects.LeftBorder())

    if hasattr(objects, "RightBorder"):
        try:
            scene_objects.append(objects.RightBorder(l=side_l))
        except TypeError:
            scene_objects.append(objects.RightBorder())

    scene_objects.append(objects.BottomBorder(bottom_a, bottom_b))

    if not hasattr(objects, "LeftBorder") and hasattr(objects, "PlinkoBorder"):
        args = data.get("plinko_border_args", [])
        scene_objects.append(objects.PlinkoBorder(*args))

    parsed_containers = []
    for idx, raw in enumerate(data.get("container_args", []) or []):
        parsed = parse_container(raw, idx)
        parsed_containers.append(parsed)
        scene_objects.append(
            objects.Container(
                Config(
                    id=idx,
                    position=(parsed["x"], parsed["y"]),
                    width=parsed["width"],
                    angle=parsed["angle"],
                )
            )
        )

    parsed_lines = []
    line_args, line_key = get_line_args_from_json(data)
    for idx, raw in enumerate(line_args):
        parsed = parse_line(raw, idx)
        parsed_lines.append(parsed)
        if not hasattr(objects, "Line"):
            raise RuntimeError("objects.Line does not exist in the uploaded project.")
        scene_objects.append(
            objects.Line(
                Config(point_a=parsed["point_a"], point_b=parsed["point_b"]),
                angle=parsed["angle"],
            )
        )

    scene_obj = Scene(
        scene_objects,
        screen_size=(RENDER_WIDTH, RENDER_HEIGHT),
    )
    scene_obj.instantiate_scene()

    static = {
        "scene_name": data.get("name", ""),
        "json_screen_width": json_width,
        "json_screen_height": json_height,
        "render_width": RENDER_WIDTH,
        "render_height": RENDER_HEIGHT,
        "fps": FPS,
        "max_event_search_frames": MAX_EVENT_SEARCH_FRAMES,
        "post_event_freeze_frames": POST_EVENT_FREEZE_FRAMES,
        "ball_start_x": ball_start[0],
        "ball_start_y": ball_start[1],
        "original_goal_x": original_goal[0],
        "original_goal_y": original_goal[1],
        "goal_y_offset_applied": goal_y_offset,
        "goal_x": goal_pos[0],
        "goal_y": goal_pos[1],
        "bottom_border_x1": bottom_a[0],
        "bottom_border_y1": bottom_a[1],
        "bottom_border_x2": bottom_b[0],
        "bottom_border_y2": bottom_b[1],
        "bottom_border_source": bottom_source,
        "side_border_l": side_l,
        "side_border_source": side_source,
        "container_count": len(parsed_containers),
        "line_count": len(parsed_lines),
        "containers_json": json.dumps(parsed_containers),
        "lines_json": json.dumps(parsed_lines),
        "objects_json": json.dumps(data.get("objects", [])),
        "full_scene_json": json.dumps(data),
        "line_args_key": line_key or "",
    }

    return scene_obj, static, parsed_lines


def get_shape_owner_map(scene_obj):
    owner = {}
    for obj in scene_obj.objects:
        for component in obj.components[1:]:
            owner[component] = getattr(obj, "name", obj.__class__.__name__)
    return owner


def get_ball_shape_and_obj(scene_obj):
    ball_obj = find_ball(scene_obj)
    for component in ball_obj.components[1:]:
        if component.__class__.__name__ == "Circle":
            return component, ball_obj
    raise RuntimeError("Could not find the ball Circle shape.")


def initial_ball_overlap_ok(scene_obj):
    ball_shape, _ = get_ball_shape_and_obj(scene_obj)
    owner = get_shape_owner_map(scene_obj)

    # Strict rule for every setting: the ball may overlap only its own shape.
    allowed_names = {"Ball"}
    bad = []

    for info in scene_obj.physics.space.shape_query(ball_shape):
        other = info.shape
        other_name = owner.get(other, "UNKNOWN")
        if other is ball_shape or other_name in allowed_names:
            continue
        bad.append(other_name)

    return len(bad) == 0, bad

def get_line_world_segment(scene_obj):
    for obj in scene_obj.objects:
        if getattr(obj, "name", "") == "Line":
            for component in obj.components[1:]:
                if component.__class__.__name__ == "Segment":
                    a = component.body.local_to_world(component.a)
                    b = component.body.local_to_world(component.b)
                    return (
                        np.array([float(a.x), float(a.y)]),
                        np.array([float(b.x), float(b.y)]),
                    )
    raise RuntimeError("Could not find a Line Segment shape.")


def point_segment_distance_and_s(points, a, b):
    points = np.asarray(points, dtype=float)
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    vector = b - a
    length = float(np.linalg.norm(vector))

    if length < 1e-6:
        distances = np.linalg.norm(points - a[None, :], axis=1)
        return distances, np.zeros(len(points)), np.zeros(len(points), dtype=bool), length

    unit = vector / length
    relative = points - a[None, :]
    s = relative @ unit
    s_clipped = np.clip(s, 0, length)
    closest = a[None, :] + s_clipped[:, None] * unit[None, :]
    distances = np.linalg.norm(points - closest, axis=1)
    on_segment = (s >= 0) & (s <= length)
    return distances, s, on_segment, length


def longest_true_run(mask):
    best = (0, None, None)
    start = None

    for idx, value in enumerate(mask):
        if value and start is None:
            start = idx
        if (not value or idx == len(mask) - 1) and start is not None:
            end = idx if value and idx == len(mask) - 1 else idx - 1
            run_length = end - start + 1
            if run_length > best[0]:
                best = (run_length, start, end)
            start = None

    return best


def line_slide_acceptance(df, line_a, line_b):
    points = df[["ball_x", "ball_y"]].to_numpy(dtype=float)
    distance, s, on_segment, line_length = point_segment_distance_and_s(
        points, line_a, line_b
    )
    near = (distance <= BALL_LINE_DISTANCE_PX) & on_segment
    run_length, run_start, run_end = longest_true_run(near)

    min_distance = float(np.nanmin(distance)) if len(distance) else float("nan")
    frames_near_total = int(np.sum(near)) if len(near) else 0

    if run_length <= 0:
        return False, {
            "line_contact_run_len": 0,
            "line_contact_start_frame": None,
            "line_contact_end_frame": None,
            "line_slide_tangential_px": 0.0,
            "line_length_px": line_length,
            "min_distance_to_line_px": min_distance,
            "frames_near_line_total": frames_near_total,
            "line_reject_reason": "never_near_line",
        }

    s_run = s[run_start : run_end + 1]
    tangential = float(np.nanmax(s_run) - np.nanmin(s_run))

    if run_length < MIN_LINE_CONTACT_CONSEC_FRAMES:
        ok, reason = False, "line_contact_run_too_short"
    elif tangential < MIN_SLIDE_TANGENTIAL_PX:
        ok, reason = False, "line_tangential_slide_too_short"
    else:
        ok, reason = True, "accepted"

    return ok, {
        "line_contact_run_len": int(run_length),
        "line_contact_start_frame": int(df.iloc[run_start]["frame"]),
        "line_contact_end_frame": int(df.iloc[run_end]["frame"]),
        "line_slide_tangential_px": tangential,
        "line_length_px": float(line_length),
        "min_distance_to_line_px": min_distance,
        "frames_near_line_total": frames_near_total,
        "line_reject_reason": reason,
    }


def reindex_scene(scene_obj):
    scene_obj.physics.space.reindex_static()
    for obj in scene_obj.objects:
        scene_obj.physics.space.reindex_shapes_for_body(obj.body)


def freeze_ball(ball_obj):
    # Stop both translation and rotation after first ground contact.
    ball_obj.body.velocity = (0.0, 0.0)
    if hasattr(ball_obj.body, "angular_velocity"):
        ball_obj.body.angular_velocity = 0.0


def simulate_fixed_window(data, goal_y_offset, setting):
    """
    Simulate until the first goal or ground event, then append 60 frozen frames.

    The first event must occur at a zero-based frame in 0..599. A collision
    caused by the step from frame f to f+1 is recorded at frame f+1.
    """
    scene_obj, static, parsed_lines = build_scene_from_data(data, goal_y_offset)
    ball_obj = find_ball(scene_obj)

    overlap_ok, overlap_names = initial_ball_overlap_ok(scene_obj)
    if not overlap_ok:
        raise RuntimeError(f"initial_ball_overlap_bad:{overlap_names}")

    if setting == "one_line":
        line_a, line_b = get_line_world_segment(scene_obj)
    else:
        line_a = line_b = None

    rows = []
    stop_event_frame = None
    stop_event_type = None
    goal_hit_frame = None
    ground_touch_frame = None

    reindex_scene(scene_obj)

    def append_state(frame, physics_frozen):
        pos = ball_obj.body.position
        vel = ball_obj.body.velocity
        rows.append(
            {
                "frame": int(frame),
                "frame_number": int(frame + 1),
                "tick": int(scene_obj.physics.tick),
                "time_seconds": frame / FPS,
                "ball_x": float(pos.x),
                "ball_y": float(pos.y),
                "ball_vx": float(vel.x),
                "ball_vy": float(vel.y),
                "physics_frozen": bool(physics_frozen),
            }
        )

    # Record natural states. If an event occurs after a forward step, append
    # that event state immediately and stop advancing physics.
    for frame in range(MAX_EVENT_SEARCH_FRAMES):
        append_state(frame, physics_frozen=False)

        if frame == MAX_EVENT_SEARCH_FRAMES - 1:
            break

        scene_obj.physics.forward()

        goal_now = bool(scene_obj.physics.collision_goal_end())
        ground_now = bool(scene_obj.physics.collision_border_end())

        if goal_now or ground_now:
            stop_event_frame = frame + 1
            goal_hit_frame = stop_event_frame if goal_now else None
            ground_touch_frame = stop_event_frame if ground_now else None

            if goal_now and ground_now:
                stop_event_type = "goal_and_ground"
            elif goal_now:
                stop_event_type = "goal"
            else:
                stop_event_type = "ground"

            freeze_ball(ball_obj)
            append_state(stop_event_frame, physics_frozen=True)
            break

    if stop_event_frame is None:
        raise RuntimeError(
            f"no_goal_or_ground_within_{MAX_EVENT_SEARCH_FRAMES}_frames"
        )

    # Add exactly 60 frames after the event frame. Physics remains frozen.
    for extra in range(1, POST_EVENT_FREEZE_FRAMES + 1):
        append_state(stop_event_frame + extra, physics_frozen=True)

    df = pd.DataFrame(rows)

    expected_output_frames = (
        stop_event_frame + 1 + POST_EVENT_FREEZE_FRAMES
    )
    if len(df) != expected_output_frames:
        raise RuntimeError(
            f"internal_frame_count_error:{len(df)}"
            f"_expected_{expected_output_frames}"
        )

    df["goal_collision_this_frame"] = (
        False
        if goal_hit_frame is None
        else df["frame"].eq(goal_hit_frame)
    )
    df["ground_collision_this_frame"] = (
        False
        if ground_touch_frame is None
        else df["frame"].eq(ground_touch_frame)
    )
    df["stop_event_this_frame"] = df["frame"].eq(stop_event_frame)

    df["goal_hit_by_frame"] = (
        False
        if goal_hit_frame is None
        else df["frame"].ge(goal_hit_frame)
    )
    df["ground_touched_by_frame"] = (
        False
        if ground_touch_frame is None
        else df["frame"].ge(ground_touch_frame)
    )
    df["stop_event_reached_by_frame"] = df["frame"].ge(stop_event_frame)

    df["goal_hit_ever"] = goal_hit_frame is not None
    df["ground_touched_ever"] = ground_touch_frame is not None
    df["stop_event_type"] = stop_event_type
    df["stop_event_frame"] = int(stop_event_frame)

    df["goal_hit_frame"] = (
        goal_hit_frame if goal_hit_frame is not None else pd.NA
    )
    df["ground_touch_frame"] = (
        ground_touch_frame if ground_touch_frame is not None else pd.NA
    )
    df["goal_hit_frame_number"] = (
        goal_hit_frame + 1 if goal_hit_frame is not None else pd.NA
    )
    df["ground_touch_frame_number"] = (
        ground_touch_frame + 1
        if ground_touch_frame is not None
        else pd.NA
    )
    df["stop_event_frame_number"] = int(stop_event_frame + 1)

    acceptance_meta = {
        "stop_event_type": stop_event_type,
        "stop_event_frame": int(stop_event_frame),
        "stop_event_frame_number": int(stop_event_frame + 1),
        "goal_hit": goal_hit_frame is not None,
        "goal_hit_frame": (
            int(goal_hit_frame) if goal_hit_frame is not None else None
        ),
        "goal_hit_frame_number": (
            int(goal_hit_frame + 1)
            if goal_hit_frame is not None
            else None
        ),
        "ground_touched": ground_touch_frame is not None,
        "ground_touch_frame": (
            int(ground_touch_frame)
            if ground_touch_frame is not None
            else None
        ),
        "ground_touch_frame_number": (
            int(ground_touch_frame + 1)
            if ground_touch_frame is not None
            else None
        ),
        "natural_simulation_frames_through_event": int(
            stop_event_frame + 1
        ),
        "post_event_padding_frames": int(POST_EVENT_FREEZE_FRAMES),
        "frozen_output_frames_including_event": int(
            POST_EVENT_FREEZE_FRAMES + 1
        ),
        "output_frames": int(len(df)),
        "frame_indexing": "frame is 0-based; frame_number is 1-based",
        "initial_ball_overlap_clear": True,
        "initial_ball_overlap_allowed_names": ["Ball"],
    }

    if setting == "one_line":
        active_df = df[df["frame"] <= stop_event_frame].copy()
        ok_slide, slide_meta = line_slide_acceptance(
            active_df, line_a, line_b
        )
        acceptance_meta.update(slide_meta)
        if not ok_slide:
            raise RuntimeError(f"line_slide_reject:{slide_meta}")
        acceptance_meta.update(
            {
                "line_world_a_x": float(line_a[0]),
                "line_world_a_y": float(line_a[1]),
                "line_world_b_x": float(line_b[0]),
                "line_world_b_y": float(line_b[1]),
            }
        )

    return scene_obj, static, df, acceptance_meta


In [ ]:
# ============================================================
# Cache-only rendering and output helpers
# ============================================================
def scene_rel_for_cache(scene_dir):
    """Return a stable path relative to the Meta-control workspace for cache IDs."""
    p = Path(scene_dir)
    try:
        return p.resolve().relative_to(BASE_DIR.resolve())
    except Exception:
        return p


def safe_scene_id(scene_dir):
    rel = scene_rel_for_cache(scene_dir)
    return str(rel).replace("/", "__").replace("\\", "__").replace(":", "")


def ball_cache_path(scene_dir):
    return BALL_CACHE_DIR / (
        f"{safe_scene_id(scene_dir)}__ball_positions_from_frames.csv"
    )


def compressed_frame_cache_path(scene_dir):
    return FRAME_CACHE_DIR / (
        f"{safe_scene_id(scene_dir)}"
        f"__frames_{IMAGE_H}x{IMAGE_W}_rgb_uint8.npy"
    )


def _shape_fill(shape, fallback):
    color = getattr(shape, "color", None)
    return tuple(color[:3]) if color else fallback


def build_render_layers(scene_obj, width, height):
    from PIL import Image, ImageDraw

    base = Image.new("RGB", (width, height), (20, 20, 24))
    base_draw = ImageDraw.Draw(base)

    for x in range(0, width, 50):
        base_draw.line([(x, 0), (x, height)], fill=(35, 35, 40))
    for y in range(0, height, 50):
        base_draw.line([(0, y), (width, y)], fill=(35, 35, 40))

    static_layer = Image.new("RGBA", (width, height), (0, 0, 0, 0))
    draw = ImageDraw.Draw(static_layer)
    ball_spec = None

    for obj in scene_obj.objects:
        is_ball = getattr(obj, "name", "") == "Ball"

        for component in obj.components[1:]:
            class_name = component.__class__.__name__

            if is_ball and class_name == "Circle":
                ball_spec = {
                    "radius": float(component.radius),
                    "fill": _shape_fill(component, (220, 40, 40)),
                    "outline": (30, 30, 30),
                    "outline_width": 2,
                }
                continue

            if class_name == "Circle":
                p = component.body.position
                radius = component.radius
                fill = _shape_fill(component, (220, 40, 40))
                draw.ellipse(
                    [p.x - radius, p.y - radius, p.x + radius, p.y + radius],
                    fill=(*fill, 255),
                    outline=(30, 30, 30, 255),
                    width=2,
                )
            elif class_name == "Segment":
                a = component.body.local_to_world(component.a)
                b = component.body.local_to_world(component.b)
                fill = _shape_fill(component, (245, 245, 245))
                draw.line(
                    [a.x, a.y, b.x, b.y],
                    fill=(*fill, 255),
                    width=max(4, int(component.radius * 2)),
                )
            elif class_name == "Poly":
                vertices = [
                    component.body.local_to_world(v)
                    for v in component.get_vertices()
                ]
                points = [(v.x, v.y) for v in vertices]
                fill = _shape_fill(component, (80, 200, 80))
                draw.polygon(
                    points,
                    fill=(*fill, 255),
                    outline=(30, 30, 30, 255),
                )

    if ball_spec is None:
        raise RuntimeError("Could not determine ball rendering specification.")

    return base, static_layer, ball_spec


def _bilinear_resample():
    from PIL import Image
    return (
        Image.Resampling.BILINEAR
        if hasattr(Image, "Resampling")
        else Image.BILINEAR
    )


def _compose_scene_without_label(
    base,
    static_layer,
    ball_spec,
    x,
    y,
):
    from PIL import ImageDraw

    radius = ball_spec["radius"]
    image = base.copy()
    draw = ImageDraw.Draw(image)
    draw.ellipse(
        [x - radius, y - radius, x + radius, y + radius],
        fill=ball_spec["fill"],
        outline=ball_spec["outline"],
        width=ball_spec["outline_width"],
    )
    image.paste(static_layer, (0, 0), static_layer)
    return image


def _render_compressed_block(
    records,
    scene_name,
    base,
    static_layer,
    ball_spec,
    stop_event_frame,
    frozen_scene_without_label,
):
    from PIL import ImageDraw

    block = np.empty(
        (len(records), 3, IMAGE_H, IMAGE_W),
        dtype=np.uint8,
    )
    frame_indices = np.empty(len(records), dtype=np.int32)
    resample = _bilinear_resample()

    for j, row in enumerate(records):
        frame = int(row["frame"])
        x = float(row["ball_x"])
        y = float(row["ball_y"])

        if frame >= int(stop_event_frame):
            image = frozen_scene_without_label.copy()
        else:
            image = _compose_scene_without_label(
                base=base,
                static_layer=static_layer,
                ball_spec=ball_spec,
                x=x,
                y=y,
            )

        draw = ImageDraw.Draw(image)
        draw.text(
            (20, 20),
            f"{scene_name}.json | frame {frame:04d}",
            fill=(230, 230, 230),
        )

        small = image.resize((IMAGE_W, IMAGE_H), resample=resample)
        arr = np.asarray(small, dtype=np.uint8)
        block[j] = np.transpose(arr, (2, 0, 1))
        frame_indices[j] = frame

    return frame_indices, block


def write_compressed_frame_cache(
    scene_obj,
    trajectory_df,
    scene_name,
    out_path,
    stop_event_frame,
):
    """
    Write one variable-length memory-mappable cache atomically.

    Cache shape is (len(trajectory_df), 3, 128, 100).
    """
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = out_path.with_name(out_path.name + ".tmp")
    tmp_path.unlink(missing_ok=True)

    base, static_layer, ball_spec = build_render_layers(
        scene_obj, RENDER_WIDTH, RENDER_HEIGHT
    )

    records = trajectory_df[
        ["frame", "ball_x", "ball_y"]
    ].to_dict("records")
    n_frames = len(records)

    event_row = trajectory_df.loc[
        trajectory_df["frame"].eq(int(stop_event_frame)),
        ["ball_x", "ball_y"],
    ]
    if len(event_row) != 1:
        raise RuntimeError(
            f"Could not identify unique stop-event frame {stop_event_frame}"
        )

    event_x = float(event_row.iloc[0]["ball_x"])
    event_y = float(event_row.iloc[0]["ball_y"])
    frozen_scene_without_label = _compose_scene_without_label(
        base=base,
        static_layer=static_layer,
        ball_spec=ball_spec,
        x=event_x,
        y=event_y,
    )

    worker_count = max(1, min(CACHE_RENDER_WORKERS, n_frames))
    chunk_size = math.ceil(n_frames / worker_count)
    chunks = [
        records[start:start + chunk_size]
        for start in range(0, n_frames, chunk_size)
    ]

    cache = np.lib.format.open_memmap(
        tmp_path,
        mode="w+",
        dtype=np.uint8,
        shape=(n_frames, 3, IMAGE_H, IMAGE_W),
    )

    try:
        with ThreadPoolExecutor(max_workers=worker_count) as executor:
            futures = [
                executor.submit(
                    _render_compressed_block,
                    chunk,
                    scene_name,
                    base,
                    static_layer,
                    ball_spec,
                    stop_event_frame,
                    frozen_scene_without_label,
                )
                for chunk in chunks
            ]

            for future in futures:
                frame_indices, block = future.result()
                cache[frame_indices] = block

        cache.flush()
        del cache
        os.replace(tmp_path, out_path)

    except Exception:
        try:
            del cache
        except Exception:
            pass
        tmp_path.unlink(missing_ok=True)
        raise

    arr = np.load(out_path, mmap_mode="r")
    expected_shape = (n_frames, 3, IMAGE_H, IMAGE_W)
    if arr.shape != expected_shape or arr.dtype != np.uint8:
        raise RuntimeError(
            f"Bad frame cache {out_path}: "
            f"shape={arr.shape}, dtype={arr.dtype}"
        )

    return out_path


def write_ball_position_cache(
    trajectory_df,
    scene_dir,
    frame_cache_path,
):
    out_path = ball_cache_path(scene_dir)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    frame_index = trajectory_df["frame"].astype(int).to_numpy()
    virtual_paths = [
        f"{frame_cache_path}::frame={int(i)}"
        for i in frame_index
    ]

    out_df = pd.DataFrame(
        {
            "frame_index": frame_index,
            "frame_path": virtual_paths,
            "ball_x": trajectory_df["ball_x"].astype(float).to_numpy(),
            "ball_y": trajectory_df["ball_y"].astype(float).to_numpy(),
        }
    )
    out_df.to_csv(out_path, index=False)

    if len(out_df) != len(trajectory_df):
        raise RuntimeError(
            f"Ball cache row count error: "
            f"{len(out_df)}/{len(trajectory_df)}"
        )

    return out_path


def cache_to_mp4(frame_cache_path, output_path):
    """Create a safety-check MP4 from a saved compressed cache."""
    import imageio.v2 as imageio
    from PIL import Image

    frame_cache_path = Path(frame_cache_path)
    output_path = Path(output_path)
    frames = np.load(frame_cache_path, mmap_mode="r")

    if (
        frames.ndim != 4
        or frames.shape[1:] != (3, IMAGE_H, IMAGE_W)
        or frames.dtype != np.uint8
    ):
        raise ValueError(
            f"Unexpected cache format: "
            f"shape={frames.shape}, dtype={frames.dtype}"
        )

    resample = (
        Image.Resampling.NEAREST
        if hasattr(Image, "Resampling")
        else Image.NEAREST
    )

    output_path.unlink(missing_ok=True)

    with imageio.get_writer(
        output_path,
        fps=FPS,
        codec="libx264",
        quality=8,
        macro_block_size=16,
        ffmpeg_params=[
            "-pix_fmt",
            "yuv420p",
            "-movflags",
            "+faststart",
        ],
    ) as writer:
        for frame_idx in range(frames.shape[0]):
            frame = np.transpose(frames[frame_idx], (1, 2, 0))
            enlarged = Image.fromarray(frame).resize(
                (PREVIEW_VIDEO_WIDTH, PREVIEW_VIDEO_HEIGHT),
                resample=resample,
            )
            writer.append_data(np.asarray(enlarged))

    print(f"Saved cache preview MP4: {output_path}")
    return output_path


def save_accepted_scene(
    setting,
    scene_name,
    data,
    source_meta,
    scene_obj,
    static,
    trajectory_df,
    acceptance_meta,
):
    setting_dir = OUTROOT / setting
    scene_dir = setting_dir / scene_name
    scene_dir.mkdir(parents=True, exist_ok=False)

    frame_cache = compressed_frame_cache_path(scene_dir)
    ball_cache = ball_cache_path(scene_dir)
    n_frames = int(len(trajectory_df))

    try:
        json_path = scene_dir / f"{scene_name}.json"
        with open(json_path, "w") as f:
            json.dump(data, f, indent=2)

        static = {
            **static,
            "setting": setting,
            "json_file": str(json_path),
            "frame_cache": str(frame_cache),
            "ball_position_cache": str(ball_cache),
            "frame_cache_shape": [n_frames, 3, IMAGE_H, IMAGE_W],
            "frame_cache_dtype": "uint8",
            **source_meta,
        }
        with open(scene_dir / "static_scene_settings.json", "w") as f:
            json.dump(static, f, indent=2)

        write_compressed_frame_cache(
            scene_obj=scene_obj,
            trajectory_df=trajectory_df,
            scene_name=scene_name,
            out_path=frame_cache,
            stop_event_frame=acceptance_meta["stop_event_frame"],
        )

        write_ball_position_cache(
            trajectory_df=trajectory_df,
            scene_dir=scene_dir,
            frame_cache_path=frame_cache,
        )

        simulation_csv_path = None
        if SAVE_SIMULATION_CSV:
            simulation_df = trajectory_df.copy()
            simulation_df["setting"] = setting
            simulation_df["scene_name"] = scene_name
            simulation_df["frame_cache_index"] = (
                simulation_df["frame"].astype(int)
            )
            simulation_df["frame_cache_path"] = str(frame_cache)
            simulation_csv_path = scene_dir / "simulation_dataset.csv"
            simulation_df.to_csv(simulation_csv_path, index=False)

        metadata = {
            "setting": setting,
            "scene_name": scene_name,
            "output_folder": str(scene_dir),
            "output_frames": n_frames,
            "fps": FPS,
            "original_render_width": RENDER_WIDTH,
            "original_render_height": RENDER_HEIGHT,
            "cache_width": IMAGE_W,
            "cache_height": IMAGE_H,
            "frame_cache_shape": [n_frames, 3, IMAGE_H, IMAGE_W],
            "frame_cache_dtype": "uint8",
            "frame_cache": str(frame_cache),
            "ball_position_cache": str(ball_cache),
            "simulation_dataset_csv": (
                str(simulation_csv_path)
                if simulation_csv_path is not None
                else None
            ),
            "json_file": str(json_path),
            "high_resolution_pngs_generated": False,
            "scene_folder_mp4_generated": False,
            "preview_mp4_generated": False,
            "preview_mp4": None,
            "pickle_generated": False,
            "npz_generated": False,
            "rendered_high_resolution_in_memory": True,
            "downsample_method": "PIL bilinear 800x1024 to 100x128",
            "cache_render_workers": CACHE_RENDER_WORKERS,
            "frozen_scene_composition_reused_after_event": True,
            "frame_label_redrawn_for_every_frame": True,
            "first_goal_or_ground_is_stopping_event": True,
            "physics_frozen_after_stop_event": True,
            **source_meta,
            **acceptance_meta,
        }

        metadata_path = scene_dir / "metadata.json"
        with open(metadata_path, "w") as f:
            json.dump(metadata, f, indent=2)

        arr = np.load(frame_cache, mmap_mode="r")
        if arr.shape != (n_frames, 3, IMAGE_H, IMAGE_W):
            raise RuntimeError(f"Unexpected frame cache shape: {arr.shape}")
        if arr.dtype != np.uint8:
            raise RuntimeError(f"Unexpected frame cache dtype: {arr.dtype}")

        ball_df = pd.read_csv(ball_cache)
        if len(ball_df) != n_frames:
            raise RuntimeError(
                f"Unexpected ball cache rows: {len(ball_df)}"
            )

        return metadata

    except Exception:
        shutil.rmtree(scene_dir, ignore_errors=True)
        frame_cache.unlink(missing_ok=True)
        ball_cache.unlink(missing_ok=True)
        raise


def add_preview_to_metadata(metadata, preview_path):
    metadata = dict(metadata)
    metadata["preview_mp4_generated"] = True
    metadata["preview_mp4"] = str(preview_path)

    metadata_path = Path(metadata["output_folder"]) / "metadata.json"
    with open(metadata_path, "w") as f:
        json.dump(metadata, f, indent=2)

    return metadata


def verify_scene_output(metadata, full_disk_check=False):
    scene_dir = Path(metadata["output_folder"])
    frame_cache = Path(metadata["frame_cache"])
    ball_cache = Path(metadata["ball_position_cache"])
    n_frames = int(metadata["output_frames"])
    stop_event_frame = int(metadata["stop_event_frame"])

    expected_n_frames = (
        stop_event_frame + 1 + POST_EVENT_FREEZE_FRAMES
    )
    if n_frames != expected_n_frames:
        raise RuntimeError(
            f"{scene_dir}: output_frames={n_frames}, "
            f"expected={expected_n_frames}"
        )

    if not bool(metadata.get("initial_ball_overlap_clear", False)):
        raise RuntimeError(f"{scene_dir}: overlap check was not recorded")
    if (scene_dir / "frames").exists():
        raise RuntimeError(
            f"{scene_dir}: unexpected high-resolution frames folder"
        )
    if (scene_dir / "scene.mp4").exists():
        raise RuntimeError(f"{scene_dir}: unexpected scene-folder MP4")

    result = {
        "scene_name": metadata["scene_name"],
        "setting": metadata["setting"],
        "frame_cache": str(frame_cache),
        "ball_cache": str(ball_cache),
        "output_frames": n_frames,
        "frame_cache_shape": str(metadata["frame_cache_shape"]),
        "frame_cache_dtype": metadata["frame_cache_dtype"],
        "ball_cache_rows": n_frames,
        "stop_event_type": metadata["stop_event_type"],
        "stop_event_frame": stop_event_frame,
        "goal_hit": bool(metadata["goal_hit"]),
        "goal_hit_frame": metadata["goal_hit_frame"],
        "ground_touched": bool(metadata["ground_touched"]),
        "ground_touch_frame": metadata["ground_touch_frame"],
        "initial_ball_overlap_clear": True,
        "high_resolution_pngs_generated": False,
        "scene_folder_mp4_generated": False,
        "preview_mp4_generated": bool(
            metadata.get("preview_mp4_generated", False)
        ),
        "preview_mp4": metadata.get("preview_mp4"),
        "full_disk_check": bool(full_disk_check),
    }

    if full_disk_check:
        arr = np.load(frame_cache, mmap_mode="r")
        positions = pd.read_csv(ball_cache)

        if arr.shape != (n_frames, 3, IMAGE_H, IMAGE_W):
            raise RuntimeError(f"{frame_cache}: bad shape {arr.shape}")
        if arr.dtype != np.uint8:
            raise RuntimeError(f"{frame_cache}: bad dtype {arr.dtype}")
        if len(positions) != n_frames:
            raise RuntimeError(
                f"{ball_cache}: bad row count {len(positions)}"
            )

        expected_indices = np.arange(n_frames)
        if not np.array_equal(
            positions["frame_index"].to_numpy(dtype=int),
            expected_indices,
        ):
            raise RuntimeError(
                f"{ball_cache}: frame indices are not consecutive"
            )

        frozen = positions.loc[
            positions["frame_index"] >= stop_event_frame,
            ["ball_x", "ball_y"],
        ].to_numpy(dtype=float)

        if len(frozen) != POST_EVENT_FREEZE_FRAMES + 1:
            raise RuntimeError(
                f"{ball_cache}: expected "
                f"{POST_EVENT_FREEZE_FRAMES + 1} event/frozen rows, "
                f"found {len(frozen)}"
            )

        if not np.allclose(frozen, frozen[0], atol=1e-6):
            raise RuntimeError(
                f"{ball_cache}: post-event ball positions are not frozen"
            )

    return result


In [ ]:
# ============================================================
# Generate all 6,000 accepted training scenes from scratch
# ============================================================
#
# The previous notebook produced 4,000 scenes and later created 2,000 more
# using separate deterministic random streams. To preserve those same random
# streams while avoiding any append/resume workflow, this notebook runs both
# generation batches inside one fresh run_generation() call.
# ============================================================

SETTING_SPECS = {
    "five_containers": {
        "builder": make_five_container_candidate,
        "goal_y_offset": EXP1_GOAL_Y_OFFSET,
        "reject_existing_exp1_match": True,
    },
    "four_containers": {
        "builder": make_four_container_candidate,
        "goal_y_offset": EXP2_GOAL_Y_OFFSET,
        "reject_existing_exp1_match": False,
    },
    "one_line": {
        "builder": make_one_line_candidate,
        "goal_y_offset": EXP1_GOAL_Y_OFFSET,
        "reject_existing_exp1_match": False,
    },
}

GENERATION_BATCHES = [
    {
        "name": "batch_1",
        "counts": {
            "five_containers": 2000,
            "four_containers": 1500,
            "one_line": 500,
        },
        "seeds": {
            "five_containers": FIVE_CONTAINER_RANDOM_SEED,
            "four_containers": FOUR_CONTAINER_RANDOM_SEED,
            "one_line": ONE_LINE_RANDOM_SEED,
        },
    },
    {
        "name": "batch_2",
        "counts": {
            "five_containers": 900,
            "four_containers": 1000,
            "one_line": 100,
        },
        "seeds": {
            "five_containers": FIVE_CONTAINER_RANDOM_SEED + 90_000_001,
            "four_containers": FOUR_CONTAINER_RANDOM_SEED + 100_000_003,
            "one_line": ONE_LINE_RANDOM_SEED + 10_000_019,
        },
    },
]

assert {
    setting: sum(batch["counts"][setting] for batch in GENERATION_BATCHES)
    for setting in SETTING_SPECS
} == TARGET_SCENE_COUNTS


def prepare_output_roots():
    roots = [OUTROOT, FRAME_CACHE_DIR, BALL_CACHE_DIR]

    if RESET_OUTPUT:
        for root in roots:
            if root.exists():
                shutil.rmtree(root)

    for root in roots:
        if root.exists() and any(root.iterdir()):
            raise RuntimeError(
                f"{root} already exists and is not empty. "
                "Set RESET_OUTPUT=True to replace it."
            )
        root.mkdir(parents=True, exist_ok=True)

    for setting in SETTING_SPECS:
        (OUTROOT / setting).mkdir(parents=True, exist_ok=True)


def run_generation():
    prepare_output_roots()

    accepted_metadata = []
    rejected_rows = []
    generated_signatures = set()
    accepted_counts = {setting: 0 for setting in SETTING_SPECS}
    start_time = time.time()

    for batch in GENERATION_BATCHES:
        batch_name = batch["name"]

        print("\n" + "#" * 90)
        print(f"Starting {batch_name}")
        print("#" * 90)

        for setting, spec in SETTING_SPECS.items():
            batch_target = int(batch["counts"][setting])
            seed = int(batch["seeds"][setting])
            rng = random.Random(seed)
            accepted_in_batch = 0
            attempts = 0

            print("\n" + "=" * 90)
            print(
                f"Generating {batch_target} {setting} scenes in {batch_name} "
                f"(final target: {TARGET_SCENE_COUNTS[setting]})"
            )
            print("=" * 90)

            while accepted_in_batch < batch_target:
                attempts += 1
                if attempts > MAX_ATTEMPTS_PER_SETTING:
                    raise RuntimeError(
                        f"Exceeded MAX_ATTEMPTS_PER_SETTING="
                        f"{MAX_ATTEMPTS_PER_SETTING} for {setting} in {batch_name}; "
                        f"accepted {accepted_in_batch}/{batch_target}."
                    )

                scene_number = accepted_counts[setting] + 1
                scene_name = f"{setting}_{scene_number:04d}"

                try:
                    data, source_meta = spec["builder"](scene_name, rng)

                    container_count = len(
                        data.get("container_args", []) or []
                    )
                    line_count = len(get_line_args_from_json(data)[0])
                    expected = {
                        "one_line": (0, 1),
                        "four_containers": (4, 0),
                        "five_containers": (5, 0),
                    }[setting]

                    if (container_count, line_count) != expected:
                        raise RuntimeError(
                            f"object_count_mismatch:"
                            f"{container_count}_containers_"
                            f"{line_count}_lines"
                        )

                    signature = canonical_scene_signature(data)

                    if (
                        spec["reject_existing_exp1_match"]
                        and signature in EXISTING_EXP1_SIGNATURES
                    ):
                        raise RuntimeError(
                            "matches_existing_experiment1_scene"
                        )

                    if signature in generated_signatures:
                        raise RuntimeError(
                            "duplicate_generated_configuration"
                        )

                    scene_obj, static, trajectory_df, acceptance_meta = (
                        simulate_fixed_window(
                            data=data,
                            goal_y_offset=spec["goal_y_offset"],
                            setting=setting,
                        )
                    )

                    metadata = save_accepted_scene(
                        setting=setting,
                        scene_name=scene_name,
                        data=data,
                        source_meta={
                            **source_meta,
                            "random_seed": seed,
                            "generation_batch": batch_name,
                            "candidate_attempt_for_batch": attempts,
                            "configuration_signature": signature,
                        },
                        scene_obj=scene_obj,
                        static=static,
                        trajectory_df=trajectory_df,
                        acceptance_meta=acceptance_meta,
                    )

                    # Only scenes 1-3 of each setting receive preview MP4s.
                    if scene_number <= PREVIEW_SCENES_PER_SETTING:
                        preview_path = (
                            BASE_DIR / f"{scene_name}_cache_preview.mp4"
                        )
                        cache_to_mp4(
                            metadata["frame_cache"],
                            preview_path,
                        )
                        metadata = add_preview_to_metadata(
                            metadata,
                            preview_path,
                        )

                    generated_signatures.add(signature)
                    accepted_metadata.append(metadata)
                    accepted_counts[setting] += 1
                    accepted_in_batch += 1

                    # Keep a recoverable live manifest during long generation.
                    pd.DataFrame(accepted_metadata).to_csv(
                        OUTROOT / "generated_scene_manifest_live.csv",
                        index=False,
                    )

                    if (
                        accepted_in_batch <= 10
                        or accepted_in_batch % 25 == 0
                        or accepted_in_batch == batch_target
                    ):
                        elapsed_min = (time.time() - start_time) / 60
                        print(
                            f"ACCEPTED {scene_name} "
                            f"({accepted_counts[setting]}/"
                            f"{TARGET_SCENE_COUNTS[setting]} total for setting; "
                            f"{accepted_in_batch}/{batch_target} in {batch_name}) "
                            f"on attempt {attempts}: "
                            f"event={acceptance_meta['stop_event_type']} "
                            f"at frame {acceptance_meta['stop_event_frame']}, "
                            f"output_frames={acceptance_meta['output_frames']}, "
                            f"elapsed={elapsed_min:.1f} min"
                        )

                except Exception as exc:
                    reason = str(exc)
                    rejected_rows.append(
                        {
                            "generation_batch": batch_name,
                            "setting": setting,
                            "attempt": attempts,
                            "prospective_scene_name": scene_name,
                            "reason": reason,
                            "traceback": traceback.format_exc(),
                        }
                    )

                    if attempts <= 10 or attempts % 100 == 0:
                        print(
                            f"Rejected {setting} {batch_name} "
                            f"attempt {attempts}: {reason}"
                        )

    for setting, target in TARGET_SCENE_COUNTS.items():
        actual = int(accepted_counts[setting])
        if actual != int(target):
            raise RuntimeError(
                f"Final count mismatch for {setting}: {actual} != {target}"
            )

    manifest = pd.DataFrame(accepted_metadata)
    manifest.to_csv(
        OUTROOT / "generated_scene_manifest.csv",
        index=False,
    )
    manifest.to_csv(
        OUTROOT / "generated_scene_manifest_live.csv",
        index=False,
    )

    rejected_df = pd.DataFrame(rejected_rows)
    rejected_df.to_csv(
        OUTROOT / "rejected_candidates.csv",
        index=False,
    )

    scene_index_columns = [
        "setting",
        "scene_name",
        "output_folder",
        "frame_cache",
        "ball_position_cache",
        "output_frames",
        "stop_event_type",
        "stop_event_frame",
        "ground_touched",
        "ground_touch_frame",
        "goal_hit",
        "goal_hit_frame",
        "preview_mp4",
        "configuration_signature",
    ]
    manifest.reindex(columns=scene_index_columns).to_csv(
        OUTROOT / "training_scene_index.csv",
        index=False,
    )

    sample_size = min(
        FULL_DISK_VERIFY_SAMPLE_SIZE,
        len(accepted_metadata),
    )
    sample_indices = (
        set(
            np.linspace(
                0,
                len(accepted_metadata) - 1,
                sample_size,
                dtype=int,
            )
        )
        if sample_size
        else set()
    )

    verification = [
        verify_scene_output(
            metadata,
            full_disk_check=(idx in sample_indices),
        )
        for idx, metadata in enumerate(accepted_metadata)
    ]
    verification_df = pd.DataFrame(verification)
    verification_df.to_csv(
        OUTROOT / "verification_summary.csv",
        index=False,
    )

    output_frame_counts = manifest["output_frames"].astype(int)
    total_cache_bytes = int(
        (output_frame_counts * 3 * IMAGE_H * IMAGE_W).sum()
    )

    summary = {
        "output_root": str(OUTROOT),
        "frame_cache_dir": str(FRAME_CACHE_DIR),
        "ball_cache_dir": str(BALL_CACHE_DIR),
        "requested_by_setting": {
            setting: int(count)
            for setting, count in TARGET_SCENE_COUNTS.items()
        },
        "requested_total": int(TOTAL_REQUESTED_SCENES),
        "accepted_total": len(accepted_metadata),
        "accepted_by_setting": (
            manifest.groupby("setting").size().astype(int).to_dict()
        ),
        "generation_batches": [
            {
                "name": batch["name"],
                "counts": {
                    setting: int(count)
                    for setting, count in batch["counts"].items()
                },
                "seeds": {
                    setting: int(seed)
                    for setting, seed in batch["seeds"].items()
                },
            }
            for batch in GENERATION_BATCHES
        ],
        "rejected_total": len(rejected_rows),
        "max_event_search_frames": MAX_EVENT_SEARCH_FRAMES,
        "post_event_freeze_frames": POST_EVENT_FREEZE_FRAMES,
        "minimum_output_frames": int(output_frame_counts.min()),
        "maximum_output_frames": int(output_frame_counts.max()),
        "mean_output_frames": float(output_frame_counts.mean()),
        "total_output_frames": int(output_frame_counts.sum()),
        "frame_cache_trailing_shape": [3, IMAGE_H, IMAGE_W],
        "frame_cache_dtype": "uint8",
        "actual_total_frame_cache_gib": (
            total_cache_bytes / (1024 ** 3)
        ),
        "first_goal_or_ground_is_stopping_event": True,
        "reject_if_no_event_within_search_window": True,
        "post_event_behavior": (
            "freeze at event position and append exactly 60 more frames"
        ),
        "initial_ball_overlap_rule": (
            "allowed_names={'Ball'} for every training setting"
        ),
        "high_resolution_pngs_generated": False,
        "scene_folder_mp4_generated": False,
        "preview_mp4s_generated": int(
            manifest["preview_mp4_generated"].sum()
        ),
        "preview_mp4_location": str(BASE_DIR),
        "pickle_generated": False,
        "npz_generated": False,
        "save_simulation_csv": bool(SAVE_SIMULATION_CSV),
        "high_resolution_rendering": (
            "in memory only, then PIL bilinear resize to 100x128"
        ),
        "cache_render_workers": CACHE_RENDER_WORKERS,
        "full_disk_verify_sample_size": int(sample_size),
        "elapsed_seconds": time.time() - start_time,
    }

    with open(OUTROOT / "generation_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print("\nGeneration complete.")
    print(json.dumps(summary, indent=2))
    return manifest, rejected_df, verification_df, summary


In [ ]:
# ============================================================
# Run all 6,000 training-scene generation
# ============================================================
manifest, rejected_candidates, verification, summary = run_generation()

display(
    manifest.groupby("setting", as_index=False)
    .agg(
        scenes=("scene_name", "size"),
        goal_stops=("goal_hit", "sum"),
        ground_stops=("ground_touched", "sum"),
        mean_stop_event_frame=("stop_event_frame", "mean"),
        mean_output_frames=("output_frames", "mean"),
    )
)

display(
    manifest[
        [
            "setting",
            "scene_name",
            "stop_event_type",
            "stop_event_frame",
            "output_frames",
            "frame_cache",
            "ball_position_cache",
            "preview_mp4",
        ]
    ].head(12)
)
display(verification.head(12))


In [ ]:
# ============================================================
# Optional variable-length cache spot check
# ============================================================
example = manifest.iloc[0]
example_frame_cache = Path(example["frame_cache"])
example_ball_cache = Path(example["ball_position_cache"])

frames = np.load(example_frame_cache, mmap_mode="r")
positions = pd.read_csv(example_ball_cache)

print("Frame cache:", example_frame_cache)
print("Shape:", frames.shape)
print("dtype:", frames.dtype)
print("Ball cache:", example_ball_cache)
print("Rows:", len(positions))
print("Stop event:", example["stop_event_type"])
print("Stop-event frame:", int(example["stop_event_frame"]))
print("Output frames:", int(example["output_frames"]))
display(positions.head())
display(positions.tail())

stop_event_frame = int(example["stop_event_frame"])
frozen_positions = positions.loc[
    positions["frame_index"] >= stop_event_frame,
    ["ball_x", "ball_y"],
].to_numpy(dtype=float)

print(
    "Event frame plus frozen padding rows:",
    len(frozen_positions),
)
print(
    "Expected event/frozen rows:",
    POST_EVENT_FREEZE_FRAMES + 1,
)
print(
    "Post-event ball positions remain fixed:",
    np.allclose(frozen_positions, frozen_positions[0], atol=1e-6),
)

print("\nScene metadata/configurations:", OUTROOT)
print("Compressed frame caches:", FRAME_CACHE_DIR)
print("Ball-position caches:", BALL_CACHE_DIR)
print(
    "\nTraining-scene discovery should use "
    f"{OUTROOT / 'training_scene_index.csv'} or the cache paths, "
    "not a frames/ directory."
)


In [ ]:

# ============================================================
# Generate original Exp1/Exp2 scenes as RNN testing data
#
# Run this after the training setup/helper cells above.
#
# It uses the same:
# - scene construction
# - Exp1/Exp2 goal offsets
# - original scene geometry exactly as configured (no initial-overlap rejection)
# - first goal/ground stopping rule
# - 60 frozen post-event frames
# - 800x1024 -> 100x128 bilinear cache pipeline
#
# It does NOT alter the existing RNN training data.
# ============================================================
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import json
import math
import os
import shutil
import time
import traceback

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Output configuration
# ------------------------------------------------------------
TEST_OUTROOT = BASE_DIR / "rnn_testing_data"

# Use the same central cache directories as the training data.
# Cache names include rnn_testing_data/exp1 or exp2, so they do
# not collide with any training cache.
TEST_FRAME_CACHE_DIR = FRAME_CACHE_DIR
TEST_BALL_CACHE_DIR = BALL_CACHE_DIR

# True starts the testing set over from scratch.
# It removes only TEST_OUTROOT and cache files whose names begin
# with rnn_testing_data__; it does not touch training scenes/caches.
RESET_TEST_OUTPUT = True

PNG_COMPRESS_LEVEL = 1
TEST_RENDER_WORKERS = min(4, os.cpu_count() or 1)

TEST_EXPERIMENTS = {
    "exp1": {
        "items": EXP1_ITEMS,
        "goal_y_offset": EXP1_GOAL_Y_OFFSET,
    },
    "exp2": {
        "items": EXP2_ITEMS,
        "goal_y_offset": EXP2_GOAL_Y_OFFSET,
    },
}


# ------------------------------------------------------------
# Confirm required helpers from the latest training generator
# ------------------------------------------------------------
required_names = [
    "HOME",
    "EXP1_ITEMS",
    "EXP2_ITEMS",
    "EXP1_GOAL_Y_OFFSET",
    "EXP2_GOAL_Y_OFFSET",
    "FRAME_CACHE_DIR",
    "BALL_CACHE_DIR",
    "IMAGE_H",
    "IMAGE_W",
    "RENDER_WIDTH",
    "RENDER_HEIGHT",
    "FPS",
    "POST_EVENT_FREEZE_FRAMES",
    "MAX_EVENT_SEARCH_FRAMES",
    "build_scene_from_data",
    "find_ball",
    "reindex_scene",
    "freeze_ball",
    "build_render_layers",
    "_compose_scene_without_label",
    "_bilinear_resample",
    "compressed_frame_cache_path",
    "ball_cache_path",
]

missing_names = [
    name for name in required_names
    if name not in globals()
]
if missing_names:
    raise RuntimeError(
        "Run the setup and helper-definition cells from "
        "generate_rnn_training_data_event60_variable_length.ipynb first. "
        f"Missing names: {missing_names}"
    )


# ------------------------------------------------------------
# Testing-only simulation: preserve original configurations
# and intentionally skip the initial-position overlap check
# ------------------------------------------------------------
def simulate_testing_scene_without_overlap_check(
    data,
    goal_y_offset,
):
    """
    Simulate an original Exp1/Exp2 configuration exactly as provided.

    Differences from the training simulator:
    - No initial ball-overlap validation or rejection is performed.
    - No randomized one-line slide-acceptance filter is performed.

    Everything else follows the event-plus-60 training rule:
    - Search for the first goal or ground collision within frames 0..599.
    - Retain the event frame.
    - Freeze the ball at the event position.
    - Append exactly POST_EVENT_FREEZE_FRAMES additional frozen frames.
    """
    scene_obj, static, _parsed_lines = build_scene_from_data(
        data,
        goal_y_offset,
    )
    ball_obj = find_ball(scene_obj)

    rows = []
    stop_event_frame = None
    stop_event_type = None
    goal_hit_frame = None
    ground_touch_frame = None

    reindex_scene(scene_obj)

    def append_state(frame, physics_frozen):
        pos = ball_obj.body.position
        vel = ball_obj.body.velocity

        rows.append(
            {
                "frame": int(frame),
                "frame_number": int(frame + 1),
                "tick": int(scene_obj.physics.tick),
                "time_seconds": frame / FPS,
                "ball_x": float(pos.x),
                "ball_y": float(pos.y),
                "ball_vx": float(vel.x),
                "ball_vy": float(vel.y),
                "physics_frozen": bool(physics_frozen),
            }
        )

    # Preserve the same event-detection timing as the training generator:
    # record frame f, advance physics, then record a collision as frame f+1.
    for frame in range(MAX_EVENT_SEARCH_FRAMES):
        append_state(
            frame,
            physics_frozen=False,
        )

        if frame == MAX_EVENT_SEARCH_FRAMES - 1:
            break

        scene_obj.physics.forward()

        goal_now = bool(
            scene_obj.physics.collision_goal_end()
        )
        ground_now = bool(
            scene_obj.physics.collision_border_end()
        )

        if goal_now or ground_now:
            stop_event_frame = frame + 1
            goal_hit_frame = (
                stop_event_frame
                if goal_now
                else None
            )
            ground_touch_frame = (
                stop_event_frame
                if ground_now
                else None
            )

            if goal_now and ground_now:
                stop_event_type = "goal_and_ground"
            elif goal_now:
                stop_event_type = "goal"
            else:
                stop_event_type = "ground"

            freeze_ball(ball_obj)
            append_state(
                stop_event_frame,
                physics_frozen=True,
            )
            break

    if stop_event_frame is None:
        raise RuntimeError(
            f"no_goal_or_ground_within_"
            f"{MAX_EVENT_SEARCH_FRAMES}_frames"
        )

    # Add exactly 60 frames after the event frame.
    for extra in range(
        1,
        POST_EVENT_FREEZE_FRAMES + 1,
    ):
        append_state(
            stop_event_frame + extra,
            physics_frozen=True,
        )

    df = pd.DataFrame(rows)

    expected_output_frames = (
        stop_event_frame
        + 1
        + POST_EVENT_FREEZE_FRAMES
    )
    if len(df) != expected_output_frames:
        raise RuntimeError(
            f"internal_frame_count_error:"
            f"{len(df)}_expected_"
            f"{expected_output_frames}"
        )

    df["goal_collision_this_frame"] = (
        False
        if goal_hit_frame is None
        else df["frame"].eq(goal_hit_frame)
    )
    df["ground_collision_this_frame"] = (
        False
        if ground_touch_frame is None
        else df["frame"].eq(
            ground_touch_frame
        )
    )
    df["stop_event_this_frame"] = (
        df["frame"].eq(stop_event_frame)
    )

    df["goal_hit_by_frame"] = (
        False
        if goal_hit_frame is None
        else df["frame"].ge(goal_hit_frame)
    )
    df["ground_touched_by_frame"] = (
        False
        if ground_touch_frame is None
        else df["frame"].ge(
            ground_touch_frame
        )
    )
    df["stop_event_reached_by_frame"] = (
        df["frame"].ge(stop_event_frame)
    )

    df["goal_hit_ever"] = (
        goal_hit_frame is not None
    )
    df["ground_touched_ever"] = (
        ground_touch_frame is not None
    )
    df["stop_event_type"] = stop_event_type
    df["stop_event_frame"] = int(
        stop_event_frame
    )

    df["goal_hit_frame"] = (
        goal_hit_frame
        if goal_hit_frame is not None
        else pd.NA
    )
    df["ground_touch_frame"] = (
        ground_touch_frame
        if ground_touch_frame is not None
        else pd.NA
    )
    df["goal_hit_frame_number"] = (
        goal_hit_frame + 1
        if goal_hit_frame is not None
        else pd.NA
    )
    df["ground_touch_frame_number"] = (
        ground_touch_frame + 1
        if ground_touch_frame is not None
        else pd.NA
    )
    df["stop_event_frame_number"] = int(
        stop_event_frame + 1
    )

    # Make the skipped validation explicit in every CSV row.
    df["initial_ball_overlap_check_performed"] = False

    acceptance_meta = {
        "stop_event_type": stop_event_type,
        "stop_event_frame": int(
            stop_event_frame
        ),
        "stop_event_frame_number": int(
            stop_event_frame + 1
        ),
        "goal_hit": (
            goal_hit_frame is not None
        ),
        "goal_hit_frame": (
            int(goal_hit_frame)
            if goal_hit_frame is not None
            else None
        ),
        "goal_hit_frame_number": (
            int(goal_hit_frame + 1)
            if goal_hit_frame is not None
            else None
        ),
        "ground_touched": (
            ground_touch_frame is not None
        ),
        "ground_touch_frame": (
            int(ground_touch_frame)
            if ground_touch_frame is not None
            else None
        ),
        "ground_touch_frame_number": (
            int(ground_touch_frame + 1)
            if ground_touch_frame is not None
            else None
        ),
        "natural_simulation_frames_through_event": int(
            stop_event_frame + 1
        ),
        "post_event_padding_frames": int(
            POST_EVENT_FREEZE_FRAMES
        ),
        "frozen_output_frames_including_event": int(
            POST_EVENT_FREEZE_FRAMES + 1
        ),
        "output_frames": int(len(df)),
        "frame_indexing": (
            "frame is 0-based; "
            "frame_number is 1-based"
        ),
        "initial_ball_overlap_check_performed": False,
        "initial_ball_overlap_clear": None,
        "initial_ball_overlap_allowed_names": None,
        "initial_ball_overlap_rule": (
            "not checked; original Exp1/Exp2 "
            "scene rendered exactly as configured"
        ),
    }

    return (
        scene_obj,
        static,
        df,
        acceptance_meta,
    )


# ------------------------------------------------------------
# Safe cleanup: remove only testing outputs
# ------------------------------------------------------------
def remove_existing_testing_outputs():
    if TEST_OUTROOT.exists():
        shutil.rmtree(TEST_OUTROOT)

    cache_prefix = f"{TEST_OUTROOT.name}__"

    for cache_dir in [
        TEST_FRAME_CACHE_DIR,
        TEST_BALL_CACHE_DIR,
    ]:
        if not cache_dir.exists():
            continue

        for path in cache_dir.iterdir():
            if path.name.startswith(cache_prefix):
                if path.is_dir():
                    shutil.rmtree(path)
                else:
                    path.unlink(missing_ok=True)


if RESET_TEST_OUTPUT:
    remove_existing_testing_outputs()

TEST_OUTROOT.mkdir(parents=True, exist_ok=True)
TEST_FRAME_CACHE_DIR.mkdir(parents=True, exist_ok=True)
TEST_BALL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

for experiment in TEST_EXPERIMENTS:
    (TEST_OUTROOT / experiment).mkdir(
        parents=True,
        exist_ok=True,
    )


# ------------------------------------------------------------
# Rendering: write high-resolution PNGs and compressed cache
# from the same in-memory image
# ------------------------------------------------------------
def _render_testing_block(
    records,
    scene_name,
    frames_dir,
    base,
    static_layer,
    ball_spec,
    stop_event_frame,
    frozen_scene_without_label,
):
    """
    Render one block.

    Returns:
      frame_indices: shape (B,)
      compressed: shape (B, 3, 128, 100), uint8

    The corresponding 800x1024 PNGs are written directly to frames_dir.
    """
    from PIL import ImageDraw

    compressed = np.empty(
        (len(records), 3, IMAGE_H, IMAGE_W),
        dtype=np.uint8,
    )
    frame_indices = np.empty(
        len(records),
        dtype=np.int32,
    )

    resample = _bilinear_resample()

    for local_index, row in enumerate(records):
        frame = int(row["frame"])
        x = float(row["ball_x"])
        y = float(row["ball_y"])

        if frame >= int(stop_event_frame):
            image = frozen_scene_without_label.copy()
        else:
            image = _compose_scene_without_label(
                base=base,
                static_layer=static_layer,
                ball_spec=ball_spec,
                x=x,
                y=y,
            )

        # Preserve the same changing label used in the training caches.
        draw = ImageDraw.Draw(image)
        draw.text(
            (20, 20),
            f"{scene_name}.json | frame {frame:04d}",
            fill=(230, 230, 230),
        )

        frame_name = f"frame_{frame:04d}.png"
        frame_path = frames_dir / frame_name

        image.save(
            frame_path,
            format="PNG",
            compress_level=PNG_COMPRESS_LEVEL,
        )

        small = image.resize(
            (IMAGE_W, IMAGE_H),
            resample=resample,
        )
        arr = np.asarray(
            small,
            dtype=np.uint8,
        )
        compressed[local_index] = np.transpose(
            arr,
            (2, 0, 1),
        )
        frame_indices[local_index] = frame

    return frame_indices, compressed


def write_testing_frames_and_cache(
    scene_obj,
    trajectory_df,
    scene_name,
    scene_dir,
    frame_cache_path,
    stop_event_frame,
):
    """
    Create:
      scene_dir/frames/frame_XXXX.png
      central compressed cache:
        (T, 3, 128, 100), uint8
    """
    scene_dir = Path(scene_dir)
    frames_dir = scene_dir / "frames"
    frames_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    frame_cache_path = Path(frame_cache_path)
    frame_cache_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp_cache = frame_cache_path.with_name(
        frame_cache_path.name + ".tmp"
    )
    tmp_cache.unlink(missing_ok=True)

    n_frames = int(len(trajectory_df))
    expected_indices = np.arange(
        n_frames,
        dtype=int,
    )

    actual_indices = trajectory_df[
        "frame"
    ].to_numpy(dtype=int)

    if not np.array_equal(
        actual_indices,
        expected_indices,
    ):
        raise RuntimeError(
            f"{scene_name}: trajectory frame indices "
            "are not consecutive from zero."
        )

    base, static_layer, ball_spec = build_render_layers(
        scene_obj,
        RENDER_WIDTH,
        RENDER_HEIGHT,
    )

    event_row = trajectory_df.loc[
        trajectory_df["frame"].eq(
            int(stop_event_frame)
        ),
        ["ball_x", "ball_y"],
    ]

    if len(event_row) != 1:
        raise RuntimeError(
            f"{scene_name}: could not find unique "
            f"stop-event frame {stop_event_frame}."
        )

    event_x = float(
        event_row.iloc[0]["ball_x"]
    )
    event_y = float(
        event_row.iloc[0]["ball_y"]
    )

    frozen_scene_without_label = (
        _compose_scene_without_label(
            base=base,
            static_layer=static_layer,
            ball_spec=ball_spec,
            x=event_x,
            y=event_y,
        )
    )

    records = trajectory_df[
        ["frame", "ball_x", "ball_y"]
    ].to_dict("records")

    worker_count = max(
        1,
        min(
            TEST_RENDER_WORKERS,
            len(records),
        ),
    )
    chunk_size = math.ceil(
        len(records) / worker_count
    )
    chunks = [
        records[start:start + chunk_size]
        for start in range(
            0,
            len(records),
            chunk_size,
        )
    ]

    cache = np.lib.format.open_memmap(
        tmp_cache,
        mode="w+",
        dtype=np.uint8,
        shape=(
            n_frames,
            3,
            IMAGE_H,
            IMAGE_W,
        ),
    )

    try:
        with ThreadPoolExecutor(
            max_workers=worker_count
        ) as executor:
            futures = [
                executor.submit(
                    _render_testing_block,
                    chunk,
                    scene_name,
                    frames_dir,
                    base,
                    static_layer,
                    ball_spec,
                    int(stop_event_frame),
                    frozen_scene_without_label,
                )
                for chunk in chunks
            ]

            for future in futures:
                indices, block = future.result()
                cache[indices] = block

        cache.flush()
        del cache
        os.replace(
            tmp_cache,
            frame_cache_path,
        )

    except Exception:
        try:
            del cache
        except Exception:
            pass

        tmp_cache.unlink(missing_ok=True)
        raise

    png_files = sorted(
        frames_dir.glob("frame_*.png")
    )
    if len(png_files) != n_frames:
        raise RuntimeError(
            f"{scene_name}: found {len(png_files)} PNGs; "
            f"expected {n_frames}."
        )

    arr = np.load(
        frame_cache_path,
        mmap_mode="r",
    )
    if (
        arr.shape
        != (
            n_frames,
            3,
            IMAGE_H,
            IMAGE_W,
        )
        or arr.dtype != np.uint8
    ):
        raise RuntimeError(
            f"{scene_name}: bad compressed cache "
            f"shape/dtype: {arr.shape}, {arr.dtype}"
        )

    return frames_dir


def write_testing_ball_cache(
    simulation_df,
    scene_dir,
):
    """
    Write the exact four columns expected by the RNN cache loader:
      frame_index, frame_path, ball_x, ball_y
    """
    out_path = ball_cache_path(
        scene_dir
    )
    out_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    ball_df = pd.DataFrame(
        {
            "frame_index": (
                simulation_df["frame"]
                .astype(int)
                .to_numpy()
            ),
            "frame_path": (
                simulation_df["frame_path"]
                .astype(str)
                .to_numpy()
            ),
            "ball_x": (
                simulation_df["ball_x"]
                .astype(float)
                .to_numpy()
            ),
            "ball_y": (
                simulation_df["ball_y"]
                .astype(float)
                .to_numpy()
            ),
        }
    )

    ball_df.to_csv(
        out_path,
        index=False,
    )

    if len(ball_df) != len(simulation_df):
        raise RuntimeError(
            "Ball-cache length does not match "
            "simulation CSV length."
        )

    return out_path


def create_local_cache_reference(
    target_path,
    local_path,
):
    """
    Make a no-extra-storage reference inside the scene folder.

    Prefer a hard link; fall back to a symbolic link.
    """
    target_path = Path(target_path)
    local_path = Path(local_path)

    if local_path.exists() or local_path.is_symlink():
        local_path.unlink()

    try:
        os.link(
            target_path,
            local_path,
        )
        return "hardlink"
    except OSError:
        local_path.symlink_to(
            target_path
        )
        return "symlink"


# ------------------------------------------------------------
# Scene completeness and cleanup helpers
# ------------------------------------------------------------
def testing_scene_is_complete(
    scene_dir,
):
    metadata_path = (
        Path(scene_dir)
        / "metadata.json"
    )
    if not metadata_path.exists():
        return False

    try:
        with open(metadata_path, "r") as f:
            metadata = json.load(f)

        n_frames = int(
            metadata["output_frames"]
        )

        required = [
            Path(metadata["json_file"]),
            Path(metadata["simulation_dataset_csv"]),
            Path(metadata["frame_cache"]),
            Path(metadata["ball_position_cache"]),
        ]
        if not all(
            path.exists()
            for path in required
        ):
            return False

        png_count = len(
            list(
                (
                    Path(scene_dir)
                    / "frames"
                ).glob("frame_*.png")
            )
        )
        if png_count != n_frames:
            return False

        arr = np.load(
            metadata["frame_cache"],
            mmap_mode="r",
        )
        if (
            arr.shape
            != (
                n_frames,
                3,
                IMAGE_H,
                IMAGE_W,
            )
            or arr.dtype != np.uint8
        ):
            return False

        ball_df = pd.read_csv(
            metadata["ball_position_cache"]
        )
        return len(ball_df) == n_frames

    except Exception:
        return False


def remove_incomplete_testing_scene(
    scene_dir,
):
    scene_dir = Path(scene_dir)

    # Compute cache paths before deleting the scene directory.
    frame_cache = compressed_frame_cache_path(
        scene_dir
    )
    ball_cache = ball_cache_path(
        scene_dir
    )

    shutil.rmtree(
        scene_dir,
        ignore_errors=True,
    )
    frame_cache.unlink(
        missing_ok=True
    )
    ball_cache.unlink(
        missing_ok=True
    )


# ------------------------------------------------------------
# Generate one original testing scene
# ------------------------------------------------------------
def generate_original_testing_scene(
    experiment,
    source_item,
    goal_y_offset,
):
    source_json = Path(
        source_item["path"]
    )
    scene_name = source_json.stem
    data = source_item["data"]

    scene_dir = (
        TEST_OUTROOT
        / experiment
        / scene_name
    )

    if scene_dir.exists():
        if testing_scene_is_complete(
            scene_dir
        ):
            with open(
                scene_dir / "metadata.json",
                "r",
            ) as f:
                metadata = json.load(f)

            print(
                f"SKIP complete: "
                f"{experiment}/{scene_name}"
            )
            return metadata

        print(
            f"Removing incomplete output: "
            f"{scene_dir}"
        )
        remove_incomplete_testing_scene(
            scene_dir
        )

    scene_dir.mkdir(
        parents=True,
        exist_ok=False,
    )

    frame_cache = (
        compressed_frame_cache_path(
            scene_dir
        )
    )
    ball_cache = ball_cache_path(
        scene_dir
    )

    try:
        # Preserve the exact original JSON file and filename.
        output_json = (
            scene_dir
            / source_json.name
        )
        shutil.copy2(
            source_json,
            output_json,
        )

        # Preserve the original scene exactly as configured.
        # No initial-position overlap check is performed, and no
        # randomized-training line-slide filter is applied.
        (
            scene_obj,
            static,
            trajectory_df,
            acceptance_meta,
        ) = simulate_testing_scene_without_overlap_check(
            data=data,
            goal_y_offset=goal_y_offset,
        )

        frames_dir = (
            write_testing_frames_and_cache(
                scene_obj=scene_obj,
                trajectory_df=trajectory_df,
                scene_name=scene_name,
                scene_dir=scene_dir,
                frame_cache_path=frame_cache,
                stop_event_frame=(
                    acceptance_meta[
                        "stop_event_frame"
                    ]
                ),
            )
        )

        simulation_df = (
            trajectory_df.copy()
        )
        n_frames = int(
            len(simulation_df)
        )

        simulation_df[
            "experiment"
        ] = experiment
        simulation_df[
            "scene_name"
        ] = scene_name
        simulation_df[
            "source_json"
        ] = str(source_json)
        simulation_df[
            "goal_y_offset_applied"
        ] = float(goal_y_offset)

        simulation_df[
            "frame_file"
        ] = [
            f"frame_{frame:04d}.png"
            for frame in simulation_df[
                "frame"
            ].astype(int)
        ]
        simulation_df[
            "frame_path"
        ] = [
            str(
                frames_dir
                / frame_file
            )
            for frame_file in simulation_df[
                "frame_file"
            ]
        ]

        simulation_df[
            "frame_cache_index"
        ] = (
            simulation_df["frame"]
            .astype(int)
        )
        simulation_df[
            "frame_cache_path"
        ] = str(frame_cache)
        simulation_df[
            "ball_position_cache_path"
        ] = str(ball_cache)

        # Repeated summary fields make each CSV self-contained.
        simulation_df[
            "total_frames"
        ] = n_frames
        simulation_df[
            "frames_saved"
        ] = n_frames
        simulation_df[
            "max_event_search_frames"
        ] = int(
            MAX_EVENT_SEARCH_FRAMES
        )
        simulation_df[
            "post_event_freeze_frames"
        ] = int(
            POST_EVENT_FREEZE_FRAMES
        )
        simulation_df[
            "natural_simulation_frames_through_event"
        ] = int(
            acceptance_meta[
                "natural_simulation_frames_through_event"
            ]
        )

        # Convenient aliases.
        simulation_df[
            "terminal_event_type"
        ] = acceptance_meta[
            "stop_event_type"
        ]
        simulation_df[
            "terminal_event_frame"
        ] = int(
            acceptance_meta[
                "stop_event_frame"
            ]
        )
        simulation_df[
            "terminal_event_frame_number"
        ] = int(
            acceptance_meta[
                "stop_event_frame_number"
            ]
        )
        simulation_df[
            "goal_touch_frame"
        ] = acceptance_meta[
            "goal_hit_frame"
        ]
        simulation_df[
            "goal_touch_frame_number"
        ] = acceptance_meta[
            "goal_hit_frame_number"
        ]

        simulation_csv = (
            scene_dir
            / "simulation_dataset.csv"
        )
        simulation_df.to_csv(
            simulation_csv,
            index=False,
        )

        # Build the exact label cache expected by the RNN notebook.
        ball_cache = (
            write_testing_ball_cache(
                simulation_df,
                scene_dir,
            )
        )

        local_frame_cache = (
            scene_dir
            / "compressed_frames_128x100_rgb_uint8.npy"
        )
        local_ball_cache = (
            scene_dir
            / "ball_positions_from_frames.csv"
        )

        frame_reference_type = (
            create_local_cache_reference(
                frame_cache,
                local_frame_cache,
            )
        )
        ball_reference_type = (
            create_local_cache_reference(
                ball_cache,
                local_ball_cache,
            )
        )

        static.update(
            {
                "experiment": experiment,
                "scene_name": scene_name,
                "source_json": str(source_json),
                "output_json": str(output_json),
                "output_folder": str(scene_dir),
                "frames_folder": str(frames_dir),
                "frame_cache": str(frame_cache),
                "ball_position_cache": str(ball_cache),
                "frame_cache_shape": [
                    n_frames,
                    3,
                    IMAGE_H,
                    IMAGE_W,
                ],
                "frame_cache_dtype": "uint8",
                "goal_y_offset_applied": float(
                    goal_y_offset
                ),
                **acceptance_meta,
            }
        )

        static_path = (
            scene_dir
            / "static_scene_settings.json"
        )
        with open(
            static_path,
            "w",
        ) as f:
            json.dump(
                static,
                f,
                indent=2,
            )

        metadata = {
            "experiment": experiment,
            "scene_name": scene_name,
            "source_json": str(source_json),
            "json_file": str(output_json),
            "output_folder": str(scene_dir),
            "frames_folder": str(frames_dir),
            "simulation_dataset_csv": str(
                simulation_csv
            ),
            "static_scene_settings": str(
                static_path
            ),
            "frame_cache": str(
                frame_cache
            ),
            "ball_position_cache": str(
                ball_cache
            ),
            "local_frame_cache_reference": str(
                local_frame_cache
            ),
            "local_ball_cache_reference": str(
                local_ball_cache
            ),
            "local_frame_cache_reference_type": (
                frame_reference_type
            ),
            "local_ball_cache_reference_type": (
                ball_reference_type
            ),
            "output_frames": n_frames,
            "frames_saved": n_frames,
            "high_resolution_frame_shape": [
                RENDER_HEIGHT,
                RENDER_WIDTH,
                3,
            ],
            "compressed_frame_cache_shape": [
                n_frames,
                3,
                IMAGE_H,
                IMAGE_W,
            ],
            "compressed_frame_cache_dtype": (
                "uint8"
            ),
            "fps": FPS,
            "high_resolution_pngs_generated": True,
            "compressed_cache_generated": True,
            "ball_position_cache_generated": True,
            "simulation_csv_generated": True,
            "mp4_generated": False,
            "first_goal_or_ground_is_stopping_event": True,
            "reject_if_no_event_within_search_window": True,
            "post_event_freeze_frames": int(
                POST_EVENT_FREEZE_FRAMES
            ),
            "initial_ball_overlap_check_performed": False,
            "initial_ball_overlap_rule": (
                "not checked; original Exp1/Exp2 "
                "scene rendered exactly as configured"
            ),
            "goal_y_offset_applied": float(
                goal_y_offset
            ),
            **acceptance_meta,
        }

        metadata_path = (
            scene_dir
            / "metadata.json"
        )
        with open(
            metadata_path,
            "w",
        ) as f:
            json.dump(
                metadata,
                f,
                indent=2,
            )

        # Final consistency checks.
        cache_array = np.load(
            frame_cache,
            mmap_mode="r",
        )
        ball_df = pd.read_csv(
            ball_cache
        )

        if (
            cache_array.shape[0] != n_frames
            or len(ball_df) != n_frames
            or len(
                list(
                    frames_dir.glob(
                        "frame_*.png"
                    )
                )
            )
            != n_frames
        ):
            raise RuntimeError(
                f"{experiment}/{scene_name}: "
                "final frame/cache/CSV counts differ."
            )

        return metadata

    except Exception:
        error_text = traceback.format_exc()

        with open(
            scene_dir / "generation_error.txt",
            "w",
        ) as f:
            f.write(error_text)

        # Do not leave cache files that could be mistaken for complete.
        frame_cache.unlink(
            missing_ok=True
        )
        ball_cache.unlink(
            missing_ok=True
        )

        raise


# ------------------------------------------------------------
# Run all original Exp1 and Exp2 configs
# ------------------------------------------------------------
start_time = time.time()
generated_metadata = []
failures = []

for experiment, spec in TEST_EXPERIMENTS.items():
    items = sorted(
        spec["items"],
        key=lambda item: item["path"].name,
    )
    total = len(items)

    print("\n" + "=" * 90)
    print(
        f"Generating original {experiment} "
        f"testing scenes: {total}"
    )
    print("=" * 90)

    for index, source_item in enumerate(
        items,
        start=1,
    ):
        scene_name = (
            source_item["path"].stem
        )

        try:
            metadata = (
                generate_original_testing_scene(
                    experiment=experiment,
                    source_item=source_item,
                    goal_y_offset=spec[
                        "goal_y_offset"
                    ],
                )
            )
            generated_metadata.append(
                metadata
            )

            print(
                f"[{experiment} {index}/{total}] "
                f"{scene_name}: "
                f"{metadata['stop_event_type']} "
                f"at frame "
                f"{metadata['stop_event_frame']}; "
                f"total={metadata['output_frames']}"
            )

        except Exception as exc:
            failures.append(
                {
                    "experiment": experiment,
                    "scene_name": scene_name,
                    "source_json": str(
                        source_item["path"]
                    ),
                    "error": str(exc),
                    "traceback": (
                        traceback.format_exc()
                    ),
                }
            )
            print(
                f"FAILED [{experiment} "
                f"{index}/{total}] "
                f"{scene_name}: {exc}"
            )


# ------------------------------------------------------------
# Save testing manifest, index, failures, and summary
# ------------------------------------------------------------
manifest = pd.DataFrame(
    generated_metadata
)

if len(manifest):
    experiment_order = {
        "exp1": 0,
        "exp2": 1,
    }
    manifest["__order"] = (
        manifest["experiment"]
        .map(experiment_order)
    )
    manifest = (
        manifest.sort_values(
            [
                "__order",
                "scene_name",
            ]
        )
        .drop(
            columns="__order"
        )
        .reset_index(drop=True)
    )

manifest_path = (
    TEST_OUTROOT
    / "testing_scene_manifest.csv"
)
manifest.to_csv(
    manifest_path,
    index=False,
)

index_columns = [
    "experiment",
    "scene_name",
    "output_folder",
    "json_file",
    "frames_folder",
    "simulation_dataset_csv",
    "frame_cache",
    "ball_position_cache",
    "output_frames",
    "stop_event_type",
    "stop_event_frame",
    "stop_event_frame_number",
    "goal_hit",
    "goal_hit_frame",
    "goal_hit_frame_number",
    "ground_touched",
    "ground_touch_frame",
    "ground_touch_frame_number",
]

manifest.reindex(
    columns=index_columns
).to_csv(
    TEST_OUTROOT
    / "testing_scene_index.csv",
    index=False,
)

failure_df = pd.DataFrame(
    failures
)
failure_df.to_csv(
    TEST_OUTROOT
    / "testing_generation_failures.csv",
    index=False,
)

source_counts = {
    experiment: len(
        spec["items"]
    )
    for experiment, spec
    in TEST_EXPERIMENTS.items()
}

generated_counts = (
    manifest.groupby(
        "experiment"
    )
    .size()
    .astype(int)
    .to_dict()
    if len(manifest)
    else {}
)

total_frames = (
    int(
        manifest[
            "output_frames"
        ].astype(int).sum()
    )
    if len(manifest)
    else 0
)

summary = {
    "testing_root": str(
        TEST_OUTROOT
    ),
    "frame_cache_dir": str(
        TEST_FRAME_CACHE_DIR
    ),
    "ball_cache_dir": str(
        TEST_BALL_CACHE_DIR
    ),
    "source_json_dirs": {
        "exp1": str(
            EXP1_JSON_DIR
        ),
        "exp2": str(
            EXP2_JSON_DIR
        ),
    },
    "source_scene_counts": (
        source_counts
    ),
    "generated_scene_counts": {
        key: int(value)
        for key, value
        in generated_counts.items()
    },
    "generated_total": int(
        len(manifest)
    ),
    "failed_total": int(
        len(failures)
    ),
    "total_output_frames": (
        total_frames
    ),
    "minimum_output_frames": (
        int(
            manifest[
                "output_frames"
            ].min()
        )
        if len(manifest)
        else None
    ),
    "maximum_output_frames": (
        int(
            manifest[
                "output_frames"
            ].max()
        )
        if len(manifest)
        else None
    ),
    "mean_output_frames": (
        float(
            manifest[
                "output_frames"
            ].mean()
        )
        if len(manifest)
        else None
    ),
    "max_event_search_frames": int(
        MAX_EVENT_SEARCH_FRAMES
    ),
    "post_event_freeze_frames": int(
        POST_EVENT_FREEZE_FRAMES
    ),
    "first_goal_or_ground_is_stopping_event": True,
    "initial_ball_overlap_check_performed": False,
    "initial_ball_overlap_rule": (
        "not checked; original Exp1/Exp2 "
        "scene rendered exactly as configured"
    ),
    "high_resolution_pngs_generated": True,
    "compressed_caches_generated": True,
    "simulation_csv_generated": True,
    "scene_names_preserved_from_original_json_stems": True,
    "cache_render_workers": int(
        TEST_RENDER_WORKERS
    ),
    "elapsed_seconds": (
        time.time() - start_time
    ),
    "manifest": str(
        manifest_path
    ),
}

with open(
    TEST_OUTROOT
    / "testing_generation_summary.json",
    "w",
) as f:
    json.dump(
        summary,
        f,
        indent=2,
    )

print("\nTesting-data generation complete.")
print(
    json.dumps(
        summary,
        indent=2,
    )
)

if failures:
    print(
        "\nWARNING: some original testing scenes failed. "
        "See testing_generation_failures.csv."
    )
else:
    print(
        "\nAll original Exp1 and Exp2 scenes "
        "were generated successfully."
    )

print(
    "\nFor the future RNN notebook, use:\n"
    f"TEST_DATA_ROOT = Path({str(TEST_OUTROOT)!r})\n"
    'TEST_EXP_FOLDERS = ["exp1", "exp2"]\n'
    f"FRAME_CACHE_DIR = Path({str(TEST_FRAME_CACHE_DIR)!r})\n"
    f"BALL_CACHE_DIR = Path({str(TEST_BALL_CACHE_DIR)!r})"
)
